# CS-781 Phase 1 — All Four Foundation Models and Five Submission Candidates

**Hardik Tiwari | Kaggle GPU T4 x2 | Four independently trained models, each 50M–51M parameters**

This complete notebook trains the full-attention decoder, encoder–decoder, local GQA decoder, and LSTM–attention hybrid as foundation runs. It then creates at least **five distinct, structurally valid prediction archives**, using two decoder decoding experiments and one experiment from each other architecture. A duplicate-content check triggers reserve experiments when needed. Competition acceptance and scores must still be obtained from your actual Codabench uploads.

**Default foundation budget: 200M supervised tokens and a five-hour cap per model per invocation.** A cap can stop a run below its token target; the notebook records that explicitly and can resume it in another invocation. The encoder–decoder learns a conditional continuation objective, while the other three learn next-token prediction over all positions. Equal supervised-token budgets therefore do not mean equal input-token budgets or compute. All models are randomly initialized unless a compatible **own** checkpoint is restored.

**Reuse your work:** the default project directory is `/kaggle/working/capstone_phase1`, matching your existing run. Your matching completed `runs/main_decoder` checkpoint is reused. Existing tokenizer, token caches, official prompts, decoder generation partials, and finished candidates are reused when their identities match. For a new Kaggle session, attach your previous `all_kaggle_outputs.zip` and set `RESTORE_OUTPUTS_ZIP` in Section 3. The archive is not embedded in this notebook.

**Source:** the [Capstone Document](https://docs.google.com/document/d/1olqoT5KPhf5CSJLBNjXGyKyyyQDBr6jcnWqPUh5cNk4/edit), through printed page 17, read 5 October 2026. Your professor's supplied Discord update permits LLM assistance for coding. The notebook includes research notes and a declaration draft; understand the code and analyze your own measured results.

**Validation:** architecture implementations retain the previously checked parameter counts, causal behavior, and cached inference. This expanded workflow is checked with CPU synthetic training, recovery, five-candidate bookkeeping, archive validation, and executable-cell syntax checks. Real official-data training and two-T4 performance require your Kaggle run; this notebook does not contain trained weights or fabricated experiment scores.

## 1. Requirements and the decisions used here

| Phase-1 requirement from pages 12-17 | Implementation |
|---|---|
| Train from scratch on the provided legal corpus | Four custom PyTorch architectures; random initialization |
| 50M-300M parameters | All four foundation models: 50.340M–50.997M |
| Explore at least four meaningfully different architectures | Full-attention decoder, encoder-decoder, local GQA decoder, recurrent-attention hybrid |
| Use the course BPE or train a phase-specific tokenizer | Train a 32,768-entry byte-level BPE on training documents only |
| No public development split | Content-hash split of whole documents; default 2% holdout |
| 512-word prompts -> 128-word continuations | Tokenize complete prompts; generate until 128 complete words are available |
| Exact official IDs and `{"ID":{"text":"..."}}` format | Preserve ID strings; reject missing/extra IDs and malformed outputs |
| `prediction.json` directly inside `submission.zip` | Archive writer and readback validator |
| Main metric BERTScore F1; secondary ROUGE-L F1 and BLEU-4 | Token-weighted local NLL plus local overlap diagnostics; official BERTScore on Codabench |
| Minimum four distinct valid Phase-1 submissions | At least five model/decoding candidates; prediction content is checked for duplicates |
| Preserve evidence and the foundation model for later phases | Logs, tokenizer, source code, best weights, recovery state, and experiment records |

The source says starter-code predictions do not count toward the submission minimum. This is a separate custom implementation. **Generating files does not establish that Codabench accepted them**: record acceptance and score after each upload. The document lists the Phase-1 deadline as **6 October 2026, 11:59 PM**; confirm the displayed competition timezone/window and leave time for scoring.

Run sections in order. The important controls are in Section 3. All four foundation runs are enabled by default; short pilots are optional and disabled. Generation targets at least five distinct candidates. These operations can take hours. Use measured timing to adjust token budgets before running long jobs.

## 2. Kaggle setup

1. Import this `.ipynb` into a new Kaggle Notebook.
2. Select **GPU T4 x2**, enable Internet, and keep the notebook private while working on this individual assignment.
3. Open the [official dataset](https://huggingface.co/datasets/Exploration-Lab/CS-781-Capstone) using your Hugging Face account and obtain any required access.
4. Add a Kaggle Secret named **`HF_TOKEN`** with a read token authorized for this dataset. Enable that secret for this notebook. It is read in memory and never printed or stored in run configs.
5. Alternatively attach the **official** Phase-1 files as a Kaggle dataset and set `LOCAL_DATA_ROOT`. The source filenames and directory structure should be retained.
6. Run the setup and preflight cells before leaving training unattended. Download/retain outputs after the run so recovery checkpoints survive a session change.

Two 15-GB GPUs do not form one 30-GB allocation. DistributedDataParallel stores a full model and optimizer on **each** GPU and splits the batch across them. At 50M parameters this is practical, while activations, attention, and vocabulary logits determine the useful micro-batch size. The notebook uses FP16 on T4; native BF16/TF32 acceleration is not assumed.

If reusing your completed Kaggle run, keep the project directory unchanged. In a fresh session attach the prior all-output archive and set `RESTORE_OUTPUTS_ZIP`; matching preprocessing can then be reused without re-downloading the corpus. The final cell again archives every working output for download.

In [ ]:
import sys, subprocess, os, json, time, math, shutil, signal, hashlib
from pathlib import Path
from IPython.display import display, Markdown, FileLink

# Keep Kaggle's installed CUDA-enabled PyTorch. Install only small helper libraries.
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet',
    'tokenizers>=0.20,<0.24', 'huggingface_hub>=0.26,<2', 'ijson>=3.2,<4',
    'requests>=2.31,<3', 'sacrebleu>=2.4,<3', 'rouge-score>=0.1.2,<0.2'])
import torch
import pandas as pd
PROJECT=Path('/kaggle/working/capstone_phase1')
PROJECT.mkdir(parents=True, exist_ok=True)
sys.path.insert(0,str(PROJECT))
os.environ['TOKENIZERS_PARALLELISM']='false'
print('Python:',sys.version.split()[0], '| PyTorch:',torch.__version__, '| CUDA build:',torch.version.cuda)
assert hasattr(torch.nn.functional,'scaled_dot_product_attention'), 'Use a PyTorch 2.x Kaggle image.'

## 3. Configuration and budgets

All four foundations default to **200M supervised tokens each**, with independent **five-hour per-invocation time caps**. The optional 5M-token pilots are disabled to avoid repeating work you already completed. A completed matching decoder run is reused. For the encoder–decoder, 200M suffix targets correspond to about one billion input token positions at the 80% prefix split, so its measured runtime differs from the full-objective decoders.

Change individual values in `FOUNDATION_TARGET_TOKENS` and `FOUNDATION_MAX_HOURS` before starting a new run. Exact resume preserves the saved optimizer schedule, token target, data identity, and batch geometry. Increasing only the allowed wall time for another invocation is supported. To change a completed run's token target/schedule, use a new run directory and an explicit own-checkpoint initialization stage rather than silently overwriting the old run.

`RESTORE_OUTPUTS_ZIP` restores your previous `/kaggle/working` tree. It rejects conflicting existing files, path traversal, and symlinks. Leave it unset if your current session already holds the files. Do not put HF tokens into configuration files; use Kaggle Secrets.

In [ ]:
SEED=2026
ALLOW_SINGLE_GPU=False                 # Set True only for an intentional one-GPU fallback.
LOCAL_DATA_ROOT=None                    # e.g. '/kaggle/input/cs781-official-capstone'
LOCAL_REVISION_LABEL='official-local-snapshot-v1'  # Change when your attached dataset version changes.
TEST_PATH_OVERRIDE=None                 # Discovery currently finds Phase_1_Pretraining/Test/test.json.
EXISTING_TOKENIZER_JSON=None             # Optional course tokenizer JSON, exactly 32,768 entries.
TOKENIZER_CHARS_PER_SOURCE=5_000_000
VALIDATION_PER_MILLE=20
SOURCE_LAYOUT={}                        # Per-source overrides: array / mapping / records / wrapper:data.
SOURCE_MIX={'acts':.10,'high_courts':.45,'supreme_court':.25,'tribunals':.20}
CACHE_TRAIN_TOKENS=240_000_000
CACHE_VALIDATION_TOKENS=2_000_000

RUN_ARCHITECTURE_SWEEP=False             # Optional pilots; the four foundation runs already explore all designs.
RUN_MAIN_TRAINING=True
RUN_GENERATION=True
MIN_SUBMISSION_CANDIDATES=5              # At least five distinct prediction files; reserves are available.
MAIN_ARCHITECTURE='decoder'             # Primary/default foundation for later phases.
FOUNDATION_ARCHITECTURES=['decoder','encoder_decoder','local_gqa','hybrid']
FOUNDATION_OBJECTIVES={'decoder':'full','encoder_decoder':'continuation','local_gqa':'full','hybrid':'full'}
FOUNDATION_TARGET_TOKENS={name:200_000_000 for name in FOUNDATION_ARCHITECTURES}
FOUNDATION_MAX_HOURS={name:5.0 for name in FOUNDATION_ARCHITECTURES}
FOUNDATION_INIT_FROM_PILOT=False         # Only your own pilot of the SAME architecture.
RUN_HELDOUT_GENERATION=True
RESTORE_OUTPUTS_ZIP=None                 # e.g. '/kaggle/input/my-phase1-outputs/all_kaggle_outputs.zip'
REUSE_EXISTING_PREPROCESSING=True
EXISTING_CACHE_DIR=None                  # Optional explicit matching cache directory.
EXISTING_OFFICIAL_PROMPTS_JSON=None      # Optional saved official prompt file.
SEQ_LEN=1024
MICRO_BATCH=4                           # Per GPU; preflight reduces this on a genuine CUDA OOM.
ACCUMULATION_STEPS=8
PILOT_TARGET_TOKENS=5_000_000
PILOT_MAX_HOURS=.5                      # Per architecture, excluding data preparation.
MAIN_TARGET_TOKENS=200_000_000
MAIN_MAX_HOURS=5.0                      # Legacy alias; per-model controls above govern foundation runs.
LEARNING_RATE=6e-4
MIN_LEARNING_RATE=6e-5
WARMUP_TOKENS=2_000_000
WEIGHT_DECAY=.1
DROPOUT=0.0
GRADIENT_CHECKPOINTING=False            # Recompute activations only if the memory tradeoff helps.
CHECKPOINT_LOSS_CHUNKS=False
LOSS_CHUNK_TOKENS=256
NUM_WORKERS=2
VALIDATION_BATCHES=64                   # Fixed subset; use a larger value for final evidence.
BENCHMARK_STEPS=12

TOKENIZER_PATH=PROJECT/'tokenizer.json'
HF_TOKEN=None
if LOCAL_DATA_ROOT is None:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN=UserSecretsClient().get_secret('HF_TOKEN')
    except Exception:
        HF_TOKEN=os.environ.get('HF_TOKEN')
    print('HF_TOKEN available:',bool(HF_TOKEN))  # Prints availability only.
WORLD_SIZE=torch.cuda.device_count()
assert torch.cuda.is_available(), 'Enable GPU T4 x2 in Kaggle settings.'
assert WORLD_SIZE==2 or (ALLOW_SINGLE_GPU and WORLD_SIZE==1), f'Found {WORLD_SIZE} GPUs; select T4 x2.'
for i in range(WORLD_SIZE):
    p=torch.cuda.get_device_properties(i)
    print(i,p.name,round(p.total_memory/2**30,2),'GiB', 'compute capability',torch.cuda.get_device_capability(i))
assert SEQ_LEN>=512 and SEQ_LEN<4096
assert sum(SOURCE_MIX.values())==1.0
print('Main full-objective target tokens/update:',WORLD_SIZE*MICRO_BATCH*ACCUMULATION_STEPS*SEQ_LEN)
print('Data-cache size:',round(2*(CACHE_TRAIN_TOKENS+CACHE_VALIDATION_TOKENS)/2**30,3),'GiB')
assert MIN_SUBMISSION_CANDIDATES>=5
assert MAIN_ARCHITECTURE in FOUNDATION_ARCHITECTURES

### Optional restore and preprocessing reuse

The next helper makes checkpoint recovery, cache reuse, and distinct-candidate bookkeeping explicit. Restore happens before model/data modules are rewritten with this notebook version.

In [ ]:
%%writefile /kaggle/working/capstone_phase1/foundation_workflow.py
"""Run planning, compatible checkpoint recovery, and distinct candidate bookkeeping."""
import hashlib
import json
from pathlib import Path
import shutil
import stat
import zipfile
import zlib

ARCHITECTURES = ('decoder', 'encoder_decoder', 'local_gqa', 'hybrid')
OBJECTIVES = {'decoder':'full', 'encoder_decoder':'continuation',
              'local_gqa':'full', 'hybrid':'full'}
IDENTITY_KEYS = ('architecture', 'target_tokens', 'seq_len', 'objective', 'prefix_fraction',
                 'seed', 'learning_rate', 'min_learning_rate', 'warmup_tokens', 'weight_decay',
                 'dropout', 'gradient_checkpointing', 'checkpoint_loss_chunks',
                 'tokenizer_sha256', 'cache_fingerprint')


def prepare_foundation_run(expected):
    """Reuse completed runs, or preserve saved batch geometry for exact recovery."""
    folder = Path(expected['output_dir'])
    config_path = folder/'run_config.json'
    summary_path = folder/'training_summary.json'
    best_path, latest_path = folder/'best_model.pt', folder/'latest.pt'
    if not config_path.exists():
        if latest_path.exists() or best_path.exists():
            raise ValueError(f'{folder}: retain run_config.json beside existing checkpoints.')
        return dict(expected), None, False
    previous = json.loads(config_path.read_text())
    for key in IDENTITY_KEYS:
        if previous.get(key) != expected.get(key):
            raise ValueError(f'{folder}: changed {key}; restore its original configuration or use a new run directory.')
    if previous.get('model_config') != expected.get('model_config'):
        raise ValueError(f'{folder}: model configuration changed.')
    if summary_path.exists():
        summary = json.loads(summary_path.read_text())
        if summary['tokens_seen'] >= expected['target_tokens']:
            if not best_path.exists():
                raise ValueError(f'{folder}: completed run is missing best_model.pt.')
            actual = dict(previous)
            for key in ('output_dir', 'cache_dir', 'tokenizer_path', 'max_hours', 'num_workers', 'cpu_threads'):
                actual[key] = expected[key]
            return actual, None, True
    if latest_path.exists():
        actual = dict(previous)
        for key in ('output_dir', 'cache_dir', 'tokenizer_path', 'max_hours', 'num_workers', 'cpu_threads'):
            actual[key] = expected[key]
        return actual, latest_path, False
    if best_path.exists():
        raise ValueError(f'{folder}: retain latest.pt for exact recovery; best_model.pt alone is insufficient.')
    return dict(expected), None, False


def select_saved_cache(project, tokenizer_hash, settings, explicit=None):
    root = Path(project)
    candidates = [Path(explicit)/'manifest.json'] if explicit else sorted((root/'cache').glob('*/manifest.json'))
    matches = []
    for path in candidates:
        if not path.exists():
            if explicit: raise FileNotFoundError(path)
            continue
        manifest = json.loads(path.read_text())
        actual = manifest.get('settings', {})
        valid = manifest.get('complete') and actual.get('tokenizer_sha256') == tokenizer_hash
        valid = valid and all(actual.get(k) == v for k,v in settings.items())
        valid = valid and all((path.parent/f'{split}.bin').is_file() and
            (path.parent/f'{split}.bin').stat().st_size == manifest[split]['tokens']*2 for split in ('train','val'))
        if valid: matches.append((path.parent,manifest))
        elif explicit: raise ValueError(f'{path}: cache is incomplete or does not match tokenizer/data settings.')
    if not matches: return None, None
    revisions = {m['settings']['revision'] for _,m in matches}
    if len(revisions)>1:
        raise ValueError('Several saved source revisions match. Set EXISTING_CACHE_DIR to the intended cache.')
    return max(matches,key=lambda item:item[1]['train']['tokens'])


def restore_outputs(archive_path, destination):
    """Extract a prior output archive; reject traversal, symlinks, and conflicting files."""
    destination = Path(destination).resolve()
    with zipfile.ZipFile(archive_path) as archive:
        members = archive.infolist()
        if len({m.filename for m in members}) != len(members):
            raise ValueError('Duplicate paths in restore archive.')
        needed = 0
        for member in members:
            path = (destination/member.filename).resolve()
            if not path.is_relative_to(destination) or stat.S_ISLNK(member.external_attr >> 16):
                raise ValueError(f'Invalid restore path: {member.filename}')
            if member.is_dir(): continue
            if path.exists():
                if not path.is_file() or path.stat().st_size != member.file_size:
                    raise ValueError(f'Conflicting existing restore file: {member.filename}')
                crc=0
                with path.open('rb') as handle:
                    for block in iter(lambda:handle.read(1024*1024),b''): crc=zlib.crc32(block,crc)
                if (crc & 0xffffffff) != member.CRC:
                    raise ValueError(f'Conflicting existing restore file: {member.filename}')
            else: needed += member.file_size
        if shutil.disk_usage(destination).free < needed + 64*1024**2:
            raise RuntimeError('Not enough free disk space to restore this output archive.')
        for member in members:
            path=destination/member.filename
            if member.is_dir(): path.mkdir(parents=True,exist_ok=True)
            elif not path.exists():
                path.parent.mkdir(parents=True,exist_ok=True)
                with archive.open(member) as src,path.open('wb') as dst:
                    shutil.copyfileobj(src,dst,1024*1024)
    return destination


def submission_plan():
    primary=[('decoder','balanced',0),('decoder','focused',0),
             ('encoder_decoder','balanced',0),('local_gqa','balanced',0),('hybrid','balanced',0)]
    reserves=[('decoder','diverse',0),('encoder_decoder','focused',0),
              ('local_gqa','focused',0),('hybrid','focused',0),('decoder','balanced',104729)]
    return [{'label':f'{arch}_{variant}'+(f'_seedplus{offset}' if offset else ''),
             'architecture':arch,'variant':variant,'seed_offset':offset,'reserve':index>=5}
            for index,(arch,variant,offset) in enumerate(primary+reserves)]


def prediction_fingerprint(predictions):
    raw=json.dumps(predictions,sort_keys=True,ensure_ascii=False,separators=(',',':')).encode()
    return hashlib.sha256(raw).hexdigest()


def add_distinct_candidate(records, candidate, predictions):
    fingerprint=prediction_fingerprint(predictions)
    duplicate=next((r['label'] for r in records if r['prediction_fingerprint']==fingerprint),None)
    if duplicate: return False, duplicate
    records.append({**candidate,'prediction_fingerprint':fingerprint,'examples':len(predictions)})
    return True,None

In [ ]:
from foundation_workflow import (prepare_foundation_run,select_saved_cache,restore_outputs,
    submission_plan,prediction_fingerprint,add_distinct_candidate)
if RESTORE_OUTPUTS_ZIP:
    restore_outputs(RESTORE_OUTPUTS_ZIP,Path('/kaggle/working'))
    print('Restored previous output files. Existing compatible work will be reused.')
SAVED_CACHE_DIR=None;SAVED_CACHE_MANIFEST=None
if REUSE_EXISTING_PREPROCESSING and TOKENIZER_PATH.exists():
    wanted={'repository':'Exploration-Lab/CS-781-Capstone','train_tokens':CACHE_TRAIN_TOKENS,
        'val_tokens':CACHE_VALIDATION_TOKENS,'mix':SOURCE_MIX,'validation_per_mille':VALIDATION_PER_MILLE,
        'layouts':SOURCE_LAYOUT,'generation_pairs':200,'tokenization_chunk_chars':32000,'version':1}
    saved_tokenizer_hash=hashlib.sha256(TOKENIZER_PATH.read_bytes()).hexdigest()
    SAVED_CACHE_DIR,SAVED_CACHE_MANIFEST=select_saved_cache(PROJECT,saved_tokenizer_hash,wanted,EXISTING_CACHE_DIR)
print('Matching preprocessing cache found:',SAVED_CACHE_DIR is not None)

## 4. Research-backed ways to spend less wall time

These ideas motivate the implementation. **Published speedups are not forecasts for two Kaggle T4s.** The benchmark later measures this exact model, vocabulary, objective, precision and context length.

| Source | What is used here | Time/memory interpretation |
|---|---|---|
| Micikevicius et al., [Mixed Precision Training](https://arxiv.org/abs/1710.03740), 2017 | FP16 autocast, FP32 model/optimizer state, GradScaler, FP32 normalization reductions | Uses T4 FP16 hardware; loss scaling protects small gradients. Measure end-to-end speed. |
| Li et al., [PyTorch Distributed](https://arxiv.org/abs/2006.15704), 2020 | One process/GPU, DDP, bucket views, `no_sync()` for the first accumulation micro-batches | Reduces repeated communication; two GPUs do not guarantee a 2x speedup. |
| Dao et al., [FlashAttention](https://arxiv.org/abs/2205.14135), 2022; Dao, [FlashAttention-2](https://arxiv.org/abs/2307.08691), 2023 | PyTorch SDPA with a causal flag for the main decoder | Avoids a hand-built attention matrix when a fused backend is supported. Standard FlashAttention-2 CUDA targets Ampere/Ada/Hopper; a separate Turing implementation exists. No `flash-attn` compilation is required here. |
| Zhang & Sennrich, [RMSNorm](https://arxiv.org/abs/1910.07467), 2019 | RMS normalization | Simpler normalization; this Python implementation is not the paper's optimized kernel benchmark. |
| Ainslie et al., [GQA](https://arxiv.org/abs/2305.13245), 2023 | Two KV heads for eight query heads in the local GQA alternative | Stores 4x fewer KV-head values; portable head repetition and a dense local mask may limit training speed. |
| Chen et al., [Sublinear Memory Training](https://arxiv.org/abs/1604.06174), 2016 | Optional block/loss checkpointing | Trades recomputation for memory. It can slow training; leave off when a useful batch fits. |
| Hoffmann et al., [Compute-Optimal LLM Training](https://arxiv.org/abs/2203.15556), 2022 | Prioritize token budget after choosing a compact legal model | The familiar roughly 20 tokens/parameter scale would be about 1B tokens for 50M. This is a broad heuristic from different settings, not a guaranteed optimum for this small legal model. 200M is a compute-limited starting budget. |

Other engineering choices: cache packed tokens once; memory-map them with PyTorch; use CUDA fused AdamW where available; avoid computing vocabulary logits for unsupervised pilot-prefix positions; run sparse validation/checkpoint events; and use KV caches plus parallel prompt shards for generation. These reduce avoidable work. Chunked cross-entropy bounds temporary FP32 conversions; without loss checkpointing its autograd state can still grow with all supervised positions.

Do not install an accelerator package merely because its paper reports good A100 results. The [official FlashAttention hardware notes](https://github.com/Dao-AILab/flash-attention#nvidia-cuda-support) and [PyTorch SDPA docs](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.scaled_dot_product_attention.html) describe backend restrictions. `torch.compile` is intentionally left off: compilation overhead and graph breaks, particularly for the LSTM alternative, should be assessed in a separate timed experiment.

## 5. Stream the official data and isolate validation documents

The next cell writes a readable helper module. It handles JSON arrays, ID-to-document mappings, JSONL records and common list wrappers. It never calls `json.load` on the giant training files. One document is parsed at a time, then tokenized in bounded whitespace-aligned pieces. Very large individual documents still require enough RAM for that single record.

Validation assignment uses SHA-256 of the whole text after case/whitespace canonicalization. Exact canonical duplicates are assigned to the same side and de-duplicated across selected sources. This prevents overlapping chunks from one document crossing into training and validation. It does **not** prove that different judgments never quote similar paragraphs; near-duplicate paragraph/case-family filtering is a useful later ablation.

Source quotas are based on **tokens**, not document counts. The `.bin` files pack documents with BOS/EOS boundaries and contain IDs in `[0,32767]`. They use two bytes/token and are read via `torch.from_file` as signed int16, then converted to `torch.long` for model input. No NumPy tensor conversion is needed.

In [ ]:
%%writefile /kaggle/working/capstone_phase1/legal_data.py
"""Stream only official Phase-1 data, split by document, train BPE, write token caches."""
import contextlib
import hashlib
import io
import json
import math
import os
from pathlib import Path
import re
import sys
from array import array
import ijson
import requests
from huggingface_hub import HfApi, hf_hub_url
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders

REPO = "Exploration-Lab/CS-781-Capstone"
SOURCE_FILES = {
    "acts": "Phase_1_Pretraining/Train/train_acts.json",
    "high_courts": "Phase_1_Pretraining/Train/train_high_courts.json",
    "supreme_court": "Phase_1_Pretraining/Train/train_supreme_court.json",
    "tribunals": "Phase_1_Pretraining/Train/train_tribunals.json",
}
TEXT_KEYS = ("text", "content", "judgment", "judgement", "document", "raw_text")
SPECIAL_TOKENS = ["<pad>", "<bos>", "<eos>", "<unk>"]


def canonical_hash(obj):
    return hashlib.sha256(json.dumps(obj, sort_keys=True, separators=(",", ":")).encode()).hexdigest()


def file_hash(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()


def atomic_json(path, obj):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(obj, indent=2, ensure_ascii=False), encoding="utf-8")
    tmp.replace(path)


def normalize_text(text):
    # Preserve case, citations, punctuation, and paragraph boundaries.
    text = text.replace("\x00", " ").replace("\r\n", "\n").replace("\r", "\n")
    return "\n".join(re.sub(r"[\t ]+", " ", line).strip() for line in text.split("\n")).strip()


def document_hash(text):
    # Whitespace/case variants receive the same train/validation assignment.
    key = " ".join(text.split()).casefold()
    return hashlib.sha256(key.encode("utf-8")).hexdigest()


def is_validation(digest, validation_per_mille=20):
    return int(digest[:16], 16) % 1000 < validation_per_mille


def extract_text(record):
    if isinstance(record, str):
        return record
    if isinstance(record, dict):
        for key in TEXT_KEYS:
            value = record.get(key)
            if isinstance(value, str):
                return value
            if isinstance(value, (dict, list)):
                found = extract_text(value)
                if found:
                    return found
        for key in ("paragraphs", "sentences"):
            if key in record:
                return extract_text(record[key])
    if isinstance(record, list):
        return "\n".join(x for x in (extract_text(v) for v in record) if x)
    return ""


@contextlib.contextmanager
def open_source(path, token=None, revision="main", local_root=None):
    if local_root:
        target = Path(local_root) / path
        if not target.exists():
            matches = list(Path(local_root).rglob(Path(path).name))
            if len(matches) != 1:
                raise FileNotFoundError(f"Expected one local file for {path}; found {len(matches)}.")
            target = matches[0]
        with open(target, "rb") as f:
            yield f
    else:
        headers = {"Authorization": f"Bearer {token}"} if token else {}
        url = hf_hub_url(REPO, path, repo_type="dataset", revision=revision)
        with requests.get(url, headers=headers, stream=True, timeout=(30, 120)) as r:
            if r.status_code in (401, 403):
                raise RuntimeError(
                    "Dataset access denied. Accept the dataset access conditions using your own "
                    "Hugging Face account, then add a Kaggle Secret named HF_TOKEN with read access."
                )
            r.raise_for_status()
            r.raw.decode_content = True
            # Pull-based parsing stops the network stream as soon as quotas are met.
            yield io.BufferedReader(r.raw, buffer_size=1 << 20)


def iter_records(stream, layout="auto"):
    if not hasattr(stream, "peek"):
        stream = io.BufferedReader(stream)
    head = stream.peek(4096).lstrip()
    if head.startswith(b"\xef\xbb\xbf"):
        stream.read(3)
        head = stream.peek(4096).lstrip()
    if layout == "auto":
        if head.startswith(b"["):
            layout = "array"
        elif head.startswith(b"{"):
            m = re.match(rb'\{\s*"([^"\\]+)"\s*:', head)
            if not m:
                raise ValueError("Unable to detect JSON layout. Set SOURCE_LAYOUT explicitly.")
            key = m.group(1).decode("utf-8")
            if key in TEXT_KEYS or key in ("id", "document_id", "doc_id", "case_id", "uid"):
                layout = "records"  # Single object or JSONL/multiple objects.
            elif key in ("data", "records", "documents", "train"):
                layout = f"wrapper:{key}"
            else:
                layout = "mapping"  # Official ID -> text / {text: ...} form.
        else:
            raise ValueError("Expected a JSON array, ID mapping, or JSONL records.")
    if layout == "array":
        yield from ijson.items(stream, "item")
    elif layout == "records":
        yield from ijson.items(stream, "", multiple_values=True)
    elif layout == "mapping":
        for _, value in ijson.kvitems(stream, ""):
            yield value
    elif layout.startswith("wrapper:"):
        key = layout.split(":", 1)[1]
        yield from ijson.items(stream, f"{key}.item")
    else:
        raise ValueError(f"Unknown JSON layout {layout}")


def iter_documents(source, token=None, revision="main", local_root=None, layout="auto"):
    bad, good = 0, 0
    with open_source(SOURCE_FILES[source], token, revision, local_root) as f:
        for record in iter_records(f, layout):
            text = normalize_text(extract_text(record))
            if not text:
                bad += 1
                if bad >= 20 and good == 0:
                    keys = list(record)[:12] if isinstance(record, dict) else type(record).__name__
                    raise ValueError(f"No legal text found in {source}; record keys/type: {keys}. Inspect the schema.")
                continue
            good += 1
            yield text, document_hash(text)
    if good == 0:
        raise ValueError(f"No documents read from {source}; verify SOURCE_LAYOUT and TEXT_KEYS.")


def resolve_revision(token=None):
    return HfApi(token=token).dataset_info(REPO).sha


def discover_test_path(token=None, revision="main", local_root=None):
    if local_root:
        paths = [str(p.relative_to(local_root)) for p in Path(local_root).rglob("*.json")
                 if "/Test/" in "/" + str(p.relative_to(local_root)) and "Phase_1" in str(p)]
    else:
        paths = [p for p in HfApi(token=token).list_repo_files(REPO, repo_type="dataset", revision=revision)
                 if p.startswith("Phase_1_Pretraining/Test/") and p.endswith(".json")]
    if len(paths) != 1:
        raise ValueError(f"Expected one Phase-1 test JSON; found {paths}. Set TEST_PATH_OVERRIDE.")
    return paths[0]


def read_test_prompts(token=None, revision="main", local_root=None, path_override=None):
    path = path_override or discover_test_path(token, revision, local_root)
    with open_source(path, token, revision, local_root) as f:
        data = json.load(f)  # Official test file is small; training files are streamed.
    if isinstance(data, dict) and all(isinstance(v, dict) and isinstance(v.get("text"), str) for v in data.values()):
        prompts = {str(k): {"text": v["text"]} for k, v in data.items()}
    elif isinstance(data, list) and all(isinstance(v, dict) and "id" in v and isinstance(v.get("text"), str) for v in data):
        ids = [str(v["id"]) for v in data]
        if len(set(ids)) != len(ids):
            raise ValueError("Duplicate official test IDs.")
        prompts = {str(v["id"]): {"text": v["text"]} for v in data}
    else:
        raise ValueError("Unknown test schema. Preserve official IDs; do not invent row-number IDs.")
    if not prompts:
        raise ValueError("Test prompts are empty.")
    return prompts, path


def train_tokenizer(output_path, token=None, revision="main", local_root=None,
                    chars_per_source=5_000_000, validation_per_mille=20, layouts=None):
    output_path = Path(output_path)
    sample_path = output_path.with_name("tokenizer_training_sample.jsonl")
    sample_path.parent.mkdir(parents=True, exist_ok=True)
    layouts = layouts or {}
    source_stats, seen = {}, set()
    with open(sample_path, "w", encoding="utf-8") as out:
        for source in SOURCE_FILES:
            chars, docs = 0, 0
            for text, digest in iter_documents(source, token, revision, local_root, layouts.get(source, "auto")):
                if is_validation(digest, validation_per_mille) or digest in seen or len(text) < 200:
                    continue
                seen.add(digest)
                portion = text[:min(chars_per_source - chars, 100_000)]
                out.write(json.dumps({"text": portion}, ensure_ascii=False) + "\n")
                chars += len(portion)
                docs += 1
                if chars >= chars_per_source:
                    break
            source_stats[source] = {"characters": chars, "documents": docs}
    def texts():
        with open(sample_path, encoding="utf-8") as f:
            for line in f:
                yield json.loads(line)["text"]
    tok = Tokenizer(models.BPE(unk_token="<unk>"))
    tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    tok.decoder = decoders.ByteLevel()
    trainer = trainers.BpeTrainer(
        vocab_size=32768, min_frequency=2, special_tokens=SPECIAL_TOKENS,
        initial_alphabet=pre_tokenizers.ByteLevel.alphabet(), show_progress=True,
    )
    tok.train_from_iterator(texts(), trainer=trainer)
    if tok.get_vocab_size() != 32768:
        raise ValueError("Tokenizer did not reach 32,768 entries. Increase chars_per_source and retrain.")
    tok.save(str(output_path))
    atomic_json(output_path.with_suffix(".provenance.json"), {
        "repository": REPO, "revision": revision, "train_only": True,
        "chars_per_source": chars_per_source,
        "validation_per_mille": validation_per_mille, "sources": source_stats,
        "sha256": file_hash(output_path), "vocab_size": tok.get_vocab_size(),
    })
    return tok


def chunks_at_whitespace(text, max_chars=32_000):
    # Bound single tokenization calls. Whitespace cuts may slightly alter BPE
    # segmentation; the same procedure is fixed for all four training caches.
    start = 0
    while start < len(text):
        end = min(len(text), start + max_chars)
        if end < len(text):
            cut = max(text.rfind(" ", start, end), text.rfind("\n", start, end))
            if cut > start + max_chars // 2:
                end = cut + 1
        yield text[start:end]
        start = end


class TokenWriter:
    def __init__(self, path):
        if sys.byteorder != "little":
            raise RuntimeError("The token cache expects a little-endian machine.")
        self.file = open(path, "wb")
        self.count, self.hash = 0, hashlib.sha256()

    def write(self, ids):
        if ids and (min(ids) < 0 or max(ids) >= 32768):
            raise ValueError("Token ID does not fit the signed-int16 cache.")
        data = array("H", ids).tobytes()
        self.file.write(data)
        self.hash.update(data)
        self.count += len(ids)

    def close(self):
        self.file.flush()
        os.fsync(self.file.fileno())
        self.file.close()


def build_cache(cache_root, tokenizer_path, token=None, revision="main", local_root=None,
                train_tokens=240_000_000, val_tokens=2_000_000,
                mix=None, validation_per_mille=20, layouts=None, generation_pairs=200):
    mix = mix or {"acts": 0.10, "high_courts": 0.45, "supreme_court": 0.25, "tribunals": 0.20}
    assert set(mix) == set(SOURCE_FILES) and abs(sum(mix.values()) - 1.0) < 1e-8
    assert all(v > 0 for v in mix.values()) and 0 < validation_per_mille < 1000
    tok = Tokenizer.from_file(str(tokenizer_path))
    assert tok.get_vocab_size() == 32768
    settings = dict(repository=REPO, revision=revision, tokenizer_sha256=file_hash(tokenizer_path),
                    train_tokens=train_tokens, val_tokens=val_tokens, mix=mix,
                    validation_per_mille=validation_per_mille, layouts=layouts or {},
                    generation_pairs=generation_pairs, tokenization_chunk_chars=32_000, version=1)
    fingerprint = canonical_hash(settings)
    root = Path(cache_root) / fingerprint[:12]
    root.mkdir(parents=True, exist_ok=True)
    manifest_path = root / "manifest.json"
    if manifest_path.exists():
        previous = json.loads(manifest_path.read_text())
        if previous.get("complete") and previous.get("fingerprint") == fingerprint:
            for split in ("train", "val"):
                assert (root / f"{split}.bin").stat().st_size == previous[split]["tokens"] * 2
            return root, previous
    # Partial caches never get marked complete and are rebuilt on the next run.
    writers = {s: TokenWriter(root / f"{s}.bin.partial") for s in ("train", "val")}
    stats, seen, pairs, audit = {}, set(), [], []
    bos, eos = tok.token_to_id("<bos>"), tok.token_to_id("<eos>")
    try:
        for source, weight in mix.items():
            quotas = {"train": math.ceil(train_tokens * weight), "val": math.ceil(val_tokens * weight)}
            counts = {"train": 0, "val": 0, "train_docs": 0, "val_docs": 0, "duplicates": 0}
            pair_quota = math.ceil(generation_pairs * weight)
            source_pairs = 0
            for text, digest in iter_documents(source, token, revision, local_root, (layouts or {}).get(source, "auto")):
                if len(text) < 200:
                    continue
                if digest in seen:
                    counts["duplicates"] += 1
                    continue
                seen.add(digest)
                split = "val" if is_validation(digest, validation_per_mille) else "train"
                if split == "val" and source_pairs < pair_quota:
                    words = text.split()
                    if len(words) >= 640:
                        pairs.append({"id": digest, "source": source,
                                      "prompt": " ".join(words[:512]), "reference": " ".join(words[512:640])})
                        source_pairs += 1
                if counts[split] >= quotas[split]:
                    if all(counts[s] >= quotas[s] for s in ("train", "val")):
                        break
                    continue
                # Hash assignment uses the whole document even when a quota cuts
                # its last cached portion. No part is moved to the other split.
                remaining = quotas[split] - counts[split]
                writers[split].write([bos])
                counts[split] += 1
                remaining -= 1
                for piece in chunks_at_whitespace(text):
                    if remaining <= 1:
                        break
                    ids = tok.encode(piece).ids[:remaining - 1]
                    writers[split].write(ids)
                    counts[split] += len(ids)
                    remaining -= len(ids)
                if remaining > 0:
                    writers[split].write([eos])
                    counts[split] += 1
                counts[f"{split}_docs"] += 1
                audit.append({"hash": digest, "source": source, "split": split})
                if all(counts[s] >= quotas[s] for s in ("train", "val")):
                    break
            if counts["train"] == 0 or counts["val"] == 0:
                raise RuntimeError(f"Insufficient train/validation documents for {source}. Increase sampling or check schema.")
            stats[source] = counts
            print(source, counts, flush=True)
    finally:
        for writer in writers.values():
            writer.close()
    for split in writers:
        (root / f"{split}.bin.partial").replace(root / f"{split}.bin")
    atomic_json(root / "validation_generation.json", pairs[:generation_pairs])
    with open(root / "document_audit.jsonl", "w") as f:
        for row in audit:
            f.write(json.dumps(row) + "\n")
    manifest = {"complete": True, "fingerprint": fingerprint, "settings": settings,
                "sources": stats, "dtype": "int16-little-endian", "vocab_size": 32768,
                "generation_pairs": min(len(pairs), generation_pairs),
                **{s: {"tokens": w.count, "sha256": w.hash.hexdigest()} for s, w in writers.items()}}
    atomic_json(manifest_path, manifest)
    return root, manifest


In [ ]:
from legal_data import (REPO,SOURCE_FILES,iter_documents,resolve_revision,read_test_prompts,
    train_tokenizer,build_cache,file_hash,atomic_json,is_validation,document_hash,canonical_hash)

REVISION=(SAVED_CACHE_MANIFEST['settings']['revision'] if SAVED_CACHE_MANIFEST else
          LOCAL_REVISION_LABEL if LOCAL_DATA_ROOT else resolve_revision(HF_TOKEN))
print('Dataset:',REPO,'| revision:',REVISION)
if SAVED_CACHE_MANIFEST:
    print('Reusing the audited official-data cache instead of streaming source documents again.')
else:
    schema_rows=[]
    for source in SOURCE_FILES:
        stream=iter_documents(source,token=HF_TOKEN,revision=REVISION,local_root=LOCAL_DATA_ROOT,
                              layout=SOURCE_LAYOUT.get(source,'auto'))
        try:
            text,digest=next(stream)
            schema_rows.append({'source':source,'first_document_characters':len(text),
                'first_document_words':len(text.split()),'hash_holdout':is_validation(digest,VALIDATION_PER_MILLE)})
        finally: stream.close()
    display(pd.DataFrame(schema_rows))
OFFICIAL_PROMPTS_PATH=PROJECT/'official_test_prompts.json'
saved_prompts=Path(EXISTING_OFFICIAL_PROMPTS_JSON) if EXISTING_OFFICIAL_PROMPTS_JSON else OFFICIAL_PROMPTS_PATH
if SAVED_CACHE_MANIFEST and saved_prompts.exists():
    OFFICIAL_PROMPTS=json.loads(saved_prompts.read_text())
    test_path='saved official_test_prompts.json from the matching previous run'
else:
    OFFICIAL_PROMPTS,test_path=read_test_prompts(token=HF_TOKEN,revision=REVISION,
        local_root=LOCAL_DATA_ROOT,path_override=TEST_PATH_OVERRIDE)
assert isinstance(OFFICIAL_PROMPTS,dict) and OFFICIAL_PROMPTS
assert all(isinstance(k,str) and isinstance(v,dict) and isinstance(v.get('text'),str)
           for k,v in OFFICIAL_PROMPTS.items())
atomic_json(OFFICIAL_PROMPTS_PATH,OFFICIAL_PROMPTS)
atomic_json(PROJECT/'data_access_metadata.json',{'repository':REPO,'revision':REVISION,
    'test_path':test_path,'test_prompts_sha256':file_hash(OFFICIAL_PROMPTS_PATH),'example_count':len(OFFICIAL_PROMPTS)})
print('Official IDs:',len(OFFICIAL_PROMPTS),'| source:',test_path)
print('Prompt word counts:',pd.Series([len(v['text'].split()) for v in OFFICIAL_PROMPTS.values()]).describe().to_dict())

## 6. Train a legal byte-level BPE tokenizer

The tokenizer sees only hash-selected **training** documents. It uses a bounded training sample from each official source and includes all byte symbols, so citations and uncommon names remain representable. It preserves case, punctuation and legal numbering. The special IDs are saved in `tokenizer.json`; the model never assumes that EOS is a specific integer.

The default vocabulary is **32,768** because it keeps the main model above the document's hard 50M minimum while allowing two-byte cached IDs. If you reuse a course tokenizer, this implementation requires that exact vocabulary size and the four named special tokens. A different vocabulary is possible, but it changes the model parameter count and cache format assumptions; rerun the audit after redesigning the dimensions.

Tokenizing a 512-word prompt does not produce 512 model tokens. Legal citations may split into many subwords. The inspection cell measures this ratio and fails if a prompt exceeds the configured generation limit, preserving the complete prompt instead of silently discarding its beginning.

In [ ]:
from tokenizers import Tokenizer
if EXISTING_TOKENIZER_JSON:
    shutil.copyfile(EXISTING_TOKENIZER_JSON,TOKENIZER_PATH)
    tokenizer=Tokenizer.from_file(str(TOKENIZER_PATH))
elif TOKENIZER_PATH.exists() and TOKENIZER_PATH.with_suffix('.provenance.json').exists():
    provenance=json.loads(TOKENIZER_PATH.with_suffix('.provenance.json').read_text())
    assert provenance['revision']==REVISION and provenance['validation_per_mille']==VALIDATION_PER_MILLE
    assert provenance.get('chars_per_source')==TOKENIZER_CHARS_PER_SOURCE, 'Tokenizer sample budget changed; use a separate tokenizer path and rebuild the cache.'
    assert provenance['sha256']==file_hash(TOKENIZER_PATH)
    tokenizer=Tokenizer.from_file(str(TOKENIZER_PATH))
    print('Reusing matching tokenizer.')
else:
    tokenizer=train_tokenizer(TOKENIZER_PATH,token=HF_TOKEN,revision=REVISION,local_root=LOCAL_DATA_ROOT,
        chars_per_source=TOKENIZER_CHARS_PER_SOURCE,validation_per_mille=VALIDATION_PER_MILLE,layouts=SOURCE_LAYOUT)
assert tokenizer.get_vocab_size()==32768
assert all(tokenizer.token_to_id(s) is not None for s in ['<pad>','<bos>','<eos>','<unk>'])
TOKENIZER_SHA256=file_hash(TOKENIZER_PATH)
print('Vocabulary:',tokenizer.get_vocab_size(),'| special IDs:',{s:tokenizer.token_to_id(s) for s in ['<pad>','<bos>','<eos>','<unk>']})
prompt_token_lengths=pd.Series([len(tokenizer.encode(v['text'].rstrip()+' ').ids) for v in OFFICIAL_PROMPTS.values()])
display(prompt_token_lengths.describe().to_frame('prompt_tokens'))
assert prompt_token_lengths.max()<=2048, 'Increase generation prompt_max_tokens/max_positions to preserve all prompt tokens.'

## 7. Build and audit the packed corpus

Cache construction can be the largest CPU/I/O task. The pipeline stops downloading a source once both quotas are met. It does not download the complete ~19-GB corpus by default. Reusing a matching completed cache avoids re-tokenization on notebook reruns.

The default 240M training-token cache is about 480MB on disk; checkpoint files are larger than the cache. `.partial` files are never treated as completed data. Completed manifests include source counts, hashes, exact token totals and a tokenizer fingerprint. When the mix, tokenizer, split or token quota changes, a different cache directory is created. Retain only the cache versions needed for your experiments if disk space becomes tight.

Packing improves utilization but allows attention across document boundaries inside a block. EOS/BOS expose those boundaries to the model. This is a common efficient corpus objective, not strict isolated-document attention. Heldout splits remain isolated because documents are assigned before packing.

In [ ]:
if SAVED_CACHE_MANIFEST:
    CACHE_DIR,cache_manifest=SAVED_CACHE_DIR,SAVED_CACHE_MANIFEST
else:
    CACHE_DIR,cache_manifest=build_cache(PROJECT/'cache',TOKENIZER_PATH,token=HF_TOKEN,revision=REVISION,
        local_root=LOCAL_DATA_ROOT,train_tokens=CACHE_TRAIN_TOKENS,val_tokens=CACHE_VALIDATION_TOKENS,
        mix=SOURCE_MIX,validation_per_mille=VALIDATION_PER_MILLE,layouts=SOURCE_LAYOUT,generation_pairs=200)
display(pd.DataFrame(cache_manifest['sources']).T)
print('Actual cache totals:',cache_manifest['train']['tokens'],'train;',cache_manifest['val']['tokens'],'validation')
assert cache_manifest['train']['tokens']>=WORLD_SIZE*MICRO_BATCH*(SEQ_LEN+1)
assert cache_manifest['val']['tokens']>=WORLD_SIZE*MICRO_BATCH*(SEQ_LEN+1)
train_hashes,val_hashes=set(),set()
with open(CACHE_DIR/'document_audit.jsonl') as f:
    for line in f:
        row=json.loads(line)
        (val_hashes if row['split']=='val' else train_hashes).add(row['hash'])
assert not (train_hashes & val_hashes)
print('Document hashes:',len(train_hashes),'train /',len(val_hashes),'validation; overlap = 0')
del train_hashes,val_hashes
VALIDATION_PAIRS=json.loads((CACHE_DIR/'validation_generation.json').read_text())
print('Heldout 512-word -> 128-word reference pairs:',len(VALIDATION_PAIRS))
assert VALIDATION_PAIRS, 'Use more official documents to obtain heldout continuation pairs.'
# SHA verification reads only compact token caches, not the 19-GB source corpus.
for split in ['train','val']:
    assert file_hash(CACHE_DIR/f'{split}.bin')==cache_manifest[split]['sha256']

## 8. Four architectures, matched near the lower size limit

| Architecture | Meaningful design difference | Trainable parameters |
|---|---|---:|
| `decoder` - main | Eight full causal self-attention blocks, RoPE, RMSNorm, SwiGLU | **50,340,352** |
| `encoder_decoder` | Three bidirectional prefix-encoder blocks, four causal decoder blocks with cross-attention | **50,341,888** |
| `local_gqa` | Nine causal blocks; eight query/two KV heads; only the most recent 256 positions are directly visible per layer | **50,996,736** |
| `hybrid` | Two unidirectional LSTM layers feeding seven causal attention blocks | **50,348,032** |

All use width 512, eight query heads, a 32,768-entry shared input/output vocabulary matrix, and SwiGLU inner width 2048. Layer counts are adjusted to keep the alternatives comparable in parameter scale. The architectures differ in information flow, not just width/depth. Local GQA is implemented with a **dense masked SDPA call**: it is a modeling/cache experiment, not an O(T-window) sparse-attention kernel. The recurrent hybrid has a sequential component, which may trade speed for its inductive bias.

For the main decoder, with vocabulary $V$, width $d$, feed-forward width $f$, and $L$ blocks:

$$N=Vd+L(4d^2+3df+2d)+d.$$

Here $V=32768$, $d=512$, $f=2048$, $L=8$, yielding **50,340,352**. The tied output head adds no second vocabulary matrix. RoPE caches are buffers and add no trainable parameters. An ordinary eight-layer model with a 32,000-entry vocabulary and these components would be **49,947,136**, below the course minimum; the size is checked explicitly rather than rounded up.

RoPE follows [Su et al.](https://arxiv.org/abs/2104.09864); SwiGLU follows [Shazeer](https://arxiv.org/abs/2002.05202). An allocated positional table of 4096 supports execution at those positions but does not prove that a model trained at 1024 extrapolates well to longer prompts. Longer-context training is an optional controlled extension below.

In [ ]:
%%writefile /kaggle/working/capstone_phase1/legal_models.py
"""Four randomly initialized legal language models; no downloaded model weights."""
import math
from dataclasses import dataclass, asdict
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.checkpoint import checkpoint


@dataclass
class ModelConfig:
    architecture: str = "decoder"
    vocab_size: int = 32768
    dim: int = 512
    heads: int = 8
    kv_heads: int = 8
    layers: int = 8
    ff_dim: int = 2048
    encoder_layers: int = 0
    rnn_layers: int = 0
    local_window: int = 0
    max_positions: int = 4096
    dropout: float = 0.0
    gradient_checkpointing: bool = False


def architecture_config(name, **overrides):
    c = ModelConfig(architecture=name)
    if name == "decoder":
        pass
    elif name == "local_gqa":
        c.kv_heads, c.layers, c.local_window = 2, 9, 256
    elif name == "hybrid":
        c.layers, c.rnn_layers = 7, 2
    elif name == "encoder_decoder":
        c.layers, c.encoder_layers = 4, 3
    else:
        raise ValueError(f"Unknown architecture: {name}")
    for k, v in overrides.items():
        if not hasattr(c, k):
            raise ValueError(k)
        setattr(c, k, v)
    assert c.dim % c.heads == 0 and c.heads % c.kv_heads == 0
    assert (c.dim // c.heads) % 2 == 0
    return c


class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        # Reduction in FP32 is useful with T4 FP16 training.
        y = x.float() * torch.rsqrt(x.float().pow(2).mean(-1, keepdim=True) + self.eps)
        return y.to(x.dtype) * self.weight.to(x.dtype)


class Rotary(nn.Module):
    def __init__(self, head_dim, max_positions):
        super().__init__()
        inv = 1.0 / (10000 ** (torch.arange(0, head_dim, 2).float() / head_dim))
        angles = torch.outer(torch.arange(max_positions).float(), inv)
        self.register_buffer("cos", angles.cos(), persistent=False)
        self.register_buffer("sin", angles.sin(), persistent=False)

    def forward(self, x, offset=0):
        n = x.size(-2)
        if offset + n > self.cos.size(0):
            raise ValueError("Position limit exceeded; increase max_positions in the model config.")
        cos = self.cos[offset:offset + n].to(x.dtype)[None, None]
        sin = self.sin[offset:offset + n].to(x.dtype)[None, None]
        a, b = x[..., 0::2], x[..., 1::2]
        return torch.stack((a * cos - b * sin, a * sin + b * cos), dim=-1).flatten(-2)


class Attention(nn.Module):
    def __init__(self, c, causal=True, cross=False):
        super().__init__()
        self.heads, self.kv_heads = c.heads, c.kv_heads
        self.hd = c.dim // c.heads
        self.q = nn.Linear(c.dim, c.dim, bias=False)
        self.k = nn.Linear(c.dim, self.kv_heads * self.hd, bias=False)
        self.v = nn.Linear(c.dim, self.kv_heads * self.hd, bias=False)
        self.out = nn.Linear(c.dim, c.dim, bias=False)
        self.rotary = None if cross else Rotary(self.hd, c.max_positions)
        self.causal, self.cross = causal, cross
        self.window = c.local_window if causal and not cross else 0
        self.dropout = c.dropout

    def forward(self, x, memory=None, past=None, position=0, use_cache=False):
        b, t, _ = x.shape
        q = self.q(x).view(b, t, self.heads, self.hd).transpose(1, 2)
        if self.cross and past is not None:
            k, v = past  # Encoder K/V projections are also cached during decoding.
        else:
            source = x if memory is None else memory
            s = source.size(1)
            k = self.k(source).view(b, s, self.kv_heads, self.hd).transpose(1, 2)
            v = self.v(source).view(b, s, self.kv_heads, self.hd).transpose(1, 2)
            if self.rotary is not None:
                k = self.rotary(k, position)
            if past is not None:
                k, v = torch.cat((past[0], k), -2), torch.cat((past[1], v), -2)
        if self.rotary is not None:
            q = self.rotary(q, position)
        new_cache = (k, v) if use_cache else None
        # Portable GQA: project/cache fewer KV heads, repeat only for the SDPA call.
        # This is not a block-sparse implementation or a guaranteed training speedup.
        if self.kv_heads != self.heads:
            repeats = self.heads // self.kv_heads
            k = k.repeat_interleave(repeats, dim=1)
            v = v.repeat_interleave(repeats, dim=1)
        mask, is_causal = None, False
        if self.causal:
            s = k.size(-2)
            if self.window or (past is not None and t > 1):
                qi = torch.arange(s - t, s, device=x.device)[:, None]
                ki = torch.arange(s, device=x.device)[None, :]
                mask = ki <= qi
                if self.window:
                    mask = mask & ((qi - ki) < self.window)
            elif past is None:
                is_causal = True
            # Cached single-token query must see all prior keys. is_causal=True
            # here would incorrectly use the upper-left non-square mask.
        y = F.scaled_dot_product_attention(
            q, k, v, attn_mask=mask, is_causal=is_causal,
            dropout_p=self.dropout if self.training else 0.0,
        )
        y = self.out(y.transpose(1, 2).contiguous().view(b, t, -1))
        return y, new_cache


class SwiGLU(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.gate = nn.Linear(c.dim, c.ff_dim, bias=False)
        self.up = nn.Linear(c.dim, c.ff_dim, bias=False)
        self.down = nn.Linear(c.ff_dim, c.dim, bias=False)

    def forward(self, x):
        return self.down(F.silu(self.gate(x)) * self.up(x))


class Block(nn.Module):
    def __init__(self, c, encoder=False):
        super().__init__()
        self.attn_norm, self.ff_norm = RMSNorm(c.dim), RMSNorm(c.dim)
        self.attn = Attention(c, causal=not encoder)
        self.ff = SwiGLU(c)
        self.drop = nn.Dropout(c.dropout)

    def forward(self, x, past=None, position=0, use_cache=False):
        y, cache = self.attn(self.attn_norm(x), past=past, position=position, use_cache=use_cache)
        x = x + self.drop(y)
        x = x + self.drop(self.ff(self.ff_norm(x)))
        return (x, cache) if use_cache else x


class CrossBlock(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.self_norm, self.cross_norm, self.ff_norm = (RMSNorm(c.dim) for _ in range(3))
        self.self_attn = Attention(c)
        self.cross_attn = Attention(c, causal=False, cross=True)
        self.ff, self.drop = SwiGLU(c), nn.Dropout(c.dropout)

    def forward(self, x, memory, past=None, cross_past=None, position=0, use_cache=False):
        y, cache = self.self_attn(self.self_norm(x), past=past, position=position, use_cache=use_cache)
        x = x + self.drop(y)
        y, cross_cache = self.cross_attn(self.cross_norm(x), memory=memory, past=cross_past, use_cache=use_cache)
        x = x + self.drop(y)
        x = x + self.drop(self.ff(self.ff_norm(x)))
        return (x, cache, cross_cache) if use_cache else x


class LegalLM(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.config = c
        self.embedding = nn.Embedding(c.vocab_size, c.dim)
        # Output projection uses embedding.weight; there is no second vocabulary matrix.
        if c.architecture == "encoder_decoder":
            self.encoder = nn.ModuleList(Block(c, encoder=True) for _ in range(c.encoder_layers))
            self.encoder_norm = RMSNorm(c.dim)
            self.blocks = nn.ModuleList(CrossBlock(c) for _ in range(c.layers))
        else:
            self.blocks = nn.ModuleList(Block(c) for _ in range(c.layers))
        if c.rnn_layers:
            self.rnn = nn.LSTM(c.dim, c.dim, num_layers=c.rnn_layers, batch_first=True)
            self.rnn_norm = RMSNorm(c.dim)
        self.final_norm = RMSNorm(c.dim)
        self.apply(self._init)
        # Scale residual projections to keep initial residual variance controlled.
        std = 0.02 / math.sqrt(2 * c.layers)
        for name, p in self.named_parameters():
            if name.endswith(("attn.out.weight", "self_attn.out.weight", "cross_attn.out.weight", "ff.down.weight")):
                nn.init.normal_(p, mean=0.0, std=std)

    @staticmethod
    def _init(m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
        elif isinstance(m, nn.LSTM):
            for name, p in m.named_parameters():
                if "weight" in name:
                    nn.init.normal_(p, mean=0.0, std=0.02)
                else:
                    nn.init.zeros_(p)

    def encode(self, source_ids):
        x = self.embedding(source_ids)
        for block in self.encoder:
            if self.config.gradient_checkpointing and self.training:
                x = checkpoint(block, x, use_reentrant=False)
            else:
                x = block(x)
        return self.encoder_norm(x)

    def forward(self, input_ids, source_ids=None, state=None, use_cache=False):
        state = state or {}
        position = state.get("position", 0)
        x = self.embedding(input_ids)
        new_state = {"position": position + input_ids.size(1), "self": []}
        if self.config.rnn_layers:
            y, recurrent_state = self.rnn(x, state.get("rnn"))
            x = x + self.rnn_norm(y)
            new_state["rnn"] = recurrent_state
        if self.config.architecture == "encoder_decoder":
            memory = state.get("memory")
            if memory is None:
                if source_ids is None:
                    raise ValueError("encoder_decoder requires source_ids on the first call.")
                memory = self.encode(source_ids)
            new_state.update(memory=memory, cross=[])
            for i, block in enumerate(self.blocks):
                if use_cache:
                    p = state.get("self", [None] * len(self.blocks))[i]
                    cp = state.get("cross", [None] * len(self.blocks))[i]
                    x, cache, cc = block(x, memory, p, cp, position, True)
                    new_state["self"].append(cache)
                    new_state["cross"].append(cc)
                elif self.config.gradient_checkpointing and self.training:
                    x = checkpoint(block, x, memory, use_reentrant=False)
                else:
                    x = block(x, memory)
        else:
            for i, block in enumerate(self.blocks):
                if use_cache:
                    p = state.get("self", [None] * len(self.blocks))[i]
                    x, cache = block(x, p, position, True)
                    new_state["self"].append(cache)
                elif self.config.gradient_checkpointing and self.training:
                    x = checkpoint(block, x, use_reentrant=False)
                else:
                    x = block(x)
        x = self.final_norm(x)
        return (x, new_state) if use_cache else x

    def project(self, hidden):
        return F.linear(hidden, self.embedding.weight)


def parameter_count(model):
    # PyTorch de-duplicates shared Parameter objects here.
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def parameter_formula(c):
    d, h, kv = c.dim, c.dim // c.heads, c.kv_heads
    attention = 2 * d * d + 2 * d * kv * h
    ff = 3 * d * c.ff_dim
    total = c.vocab_size * d + d
    if c.architecture == "encoder_decoder":
        total += c.encoder_layers * (attention + ff + 2 * d) + d
        total += c.layers * (2 * attention + ff + 3 * d)
    else:
        total += c.layers * (attention + ff + 2 * d)
    if c.rnn_layers:
        total += c.rnn_layers * (8 * d * d + 8 * d) + d
    return total


def make_batch(blocks, architecture, prefix_fraction=0.80, objective="full"):
    # blocks has T+1 tokens. The target suffix is identical across architectures.
    t = blocks.size(1) - 1
    split = max(1, min(t, int(t * prefix_fraction)))
    if architecture == "encoder_decoder":
        source = blocks[:, :split]
        inputs, targets = blocks[:, split - 1:-1], blocks[:, split:]
    else:
        source = None
        inputs, targets = blocks[:, :-1], blocks[:, 1:].clone()
        if objective == "continuation":
            targets[:, :split - 1] = -100
    return inputs, targets, source


def language_model_loss(hidden, targets, weight, chunk_tokens=256, checkpoint_chunks=False):
    """Sum/token count lets validation aggregate the true token-weighted NLL."""
    hidden, targets = hidden.reshape(-1, hidden.size(-1)), targets.reshape(-1)
    total = hidden.new_zeros((), dtype=torch.float32)
    count = (targets != -100).sum()
    def loss_chunk(h, y, w):
        # Keep logits in autocast precision until the bounded FP32 CE conversion.
        return F.cross_entropy(F.linear(h, w).float(), y, ignore_index=-100, reduction="sum")
    for i in range(0, targets.numel(), chunk_tokens):
        h, y = hidden[i:i + chunk_tokens], targets[i:i + chunk_tokens]
        if checkpoint_chunks and torch.is_grad_enabled():
            total = total + checkpoint(loss_chunk, h, y, weight, use_reentrant=False)
        else:
            total = total + loss_chunk(h, y, weight)
    return total / count.clamp_min(1), total.detach(), count.detach()


if __name__ == "__main__":
    for name in ("decoder", "encoder_decoder", "local_gqa", "hybrid"):
        c = architecture_config(name)
        with torch.device("meta"):
            model = LegalLM(c)
        n = parameter_count(model)
        assert n == parameter_formula(c) and 50_000_000 <= n <= 300_000_000
        print(name, f"{n:,}", asdict(c))


In [ ]:
from dataclasses import asdict
from legal_models import (LegalLM,architecture_config,parameter_count,parameter_formula,
                         make_batch,language_model_loss)
ARCHITECTURES=['decoder','encoder_decoder','local_gqa','hybrid']
architecture_rows=[]
for name in ARCHITECTURES:
    config=architecture_config(name)
    with torch.device('meta'):
        model=LegalLM(config)
    n=parameter_count(model)
    assert n==parameter_formula(config)
    assert 50_000_000<=n<=300_000_000
    architecture_rows.append({'architecture':name,'parameters':n,'millions':n/1e6})
display(pd.DataFrame(architecture_rows))
del model

## 9. Numerical smoke checks before expensive training

Small **test-only** configurations check causality, cache equivalence and finite gradients. These models are below 50M and are **never used for official predictions**. They make a numerical bug cheap to find before a long Kaggle run. Actual training launchers independently enforce the 50M-300M bound.

The encoder may attend bidirectionally to the prefix because that prefix is given to the model. It never sees suffix targets. The decoder's shifted suffix input and causal mask prevent future-target leakage. Cached single-token attention uses all previous keys; applying the square causal mask naively to a one-token query would incorrectly hide most of the cache.

In [ ]:
torch.manual_seed(SEED)
for name in ARCHITECTURES:
    cfg=architecture_config(name,vocab_size=128,dim=32,heads=4,kv_heads=2 if name=='local_gqa' else 4,
        ff_dim=64,layers=2,encoder_layers=1 if name=='encoder_decoder' else 0,
        local_window=8 if name=='local_gqa' else 0,max_positions=256)
    tiny=LegalLM(cfg).eval()
    ids=torch.randint(4,128,(2,17))
    src=torch.randint(4,128,(2,11)) if name=='encoder_decoder' else None
    with torch.no_grad():
        full=tiny(ids,source_ids=src)
        states=None;pieces=[]
        for i in range(ids.size(1)):
            h,states=tiny(ids[:,i:i+1],source_ids=src if i==0 else None,state=states,use_cache=True)
            pieces.append(h)
        cache_error=(full-torch.cat(pieces,1)).abs().max().item()
        changed=ids.clone();changed[:,10:]=torch.randint(4,128,changed[:,10:].shape)
        causal_error=(tiny(changed,source_ids=src)[:,:10]-full[:,:10]).abs().max().item()
    assert cache_error<2e-5 and causal_error<2e-5
    tiny.train();tiny.config.gradient_checkpointing=True
    x,y,s=make_batch(ids,name,objective='continuation')
    loss,_,_=language_model_loss(tiny(x,source_ids=s),y,tiny.embedding.weight,
                                chunk_tokens=8,checkpoint_chunks=True)
    loss.backward()
    assert torch.isfinite(loss) and all(p.grad is not None and torch.isfinite(p.grad).all() for p in tiny.parameters())
    print(name,'passed; cache max error:',cache_error,'causality max error:',causal_error)
del tiny,full,pieces,states,loss

## 10. Training objective and validation protocol

For a causal decoder, $p(x_t|x_{<t})$ is learned with next-token cross-entropy. Main pretraining supervises all positions. For the controlled architecture pilots, the first 80% of block positions are context and the suffix is predicted, approximating the competition's 512:128 word ratio in token space. The encoder-decoder receives only that prefix in its encoder, then uses the last prefix token followed by shifted suffix tokens as decoder input. The other architectures receive the complete causal sequence but compute loss only on the same suffix target positions. Exact word-boundary continuation examples are assessed separately during generation.

Validation always evaluates this **common conditional suffix NLL**, even for a main decoder trained on all positions. It aggregates summed loss divided by the number of supervised tokens across both GPUs. The reported perplexity is $\exp(\mathrm{NLL})$ for this conditional objective. It is not the whole-corpus full-token perplexity or a substitute for BERTScore. A different tokenizer, validation sample or context length makes NLL values less directly comparable.

Main defaults: AdamW, LR $6\times10^{-4}$ to $6\times10^{-5}$, short token-based warmup, cosine decay, weight decay 0.1 for matrices, no decay on vectors, and gradient clipping at 1.0. These are starting values to test. Effective batch size is **micro-batch x GPUs x accumulation**, and FP16 gradients are unscaled before clipping. Gradients are synchronized only on the final accumulation micro-batch.

`best_model.pt` is selected by measured validation NLL. `latest.pt` additionally retains optimizer, scaler, RNG state per rank, sampler epoch and consumed batch offset. Exact resume requires the same model, data, batch geometry, objective and schedule. A deliberate schedule/context change uses `--init-from` with fresh optimizer state and is logged as another training stage.

In [ ]:
%%writefile /kaggle/working/capstone_phase1/train_phase1.py
"""Run under torchrun: one process/GPU, FP16, correct gradient accumulation, recovery."""
import argparse
import contextlib
import csv
from datetime import timedelta
import inspect
import json
import math
import os
from pathlib import Path
import random
import signal
import shutil
import time
from dataclasses import asdict
import torch
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import Dataset, DataLoader, DistributedSampler
from legal_models import LegalLM, ModelConfig, architecture_config, parameter_count, make_batch, language_model_loss
from legal_data import atomic_json, canonical_hash, file_hash


class TokenBlocks(Dataset):
    def __init__(self, path, seq_len):
        self.path, self.seq_len = str(path), seq_len
        self.tokens = Path(path).stat().st_size // 2
        self.mapped = None

    def __len__(self):
        return (self.tokens - 1) // self.seq_len

    def __getitem__(self, index):
        if self.mapped is None:
            self.mapped = torch.from_file(self.path, shared=False, size=self.tokens, dtype=torch.int16)
        start = int(index) * self.seq_len
        return self.mapped[start:start + self.seq_len + 1].to(torch.long)


def setup_distributed():
    world, rank, local_rank = (int(os.environ.get(k, default)) for k, default in
                              (("WORLD_SIZE", 1), ("RANK", 0), ("LOCAL_RANK", 0)))
    if torch.cuda.is_available():
        torch.cuda.set_device(local_rank)
        device = torch.device("cuda", local_rank)
    else:
        device = torch.device("cpu")
    if world > 1:
        dist.init_process_group("nccl" if device.type == "cuda" else "gloo", timeout=timedelta(minutes=30))
    return device, world, rank


def synchronize(device):
    if device.type == "cuda":
        torch.cuda.synchronize(device)


def rng_state(device):
    return {"python": random.getstate(), "torch": torch.get_rng_state(),
            "cuda": torch.cuda.get_rng_state(device) if device.type == "cuda" else None}


def restore_rng(state, device):
    random.setstate(state["python"])
    torch.set_rng_state(state["torch"].cpu())
    if state["cuda"] is not None and device.type == "cuda":
        torch.cuda.set_rng_state(state["cuda"].cpu(), device)


def crop_loss_view(hidden, targets, architecture, prefix_fraction, objective, seq_len):
    if objective == "continuation" and architecture != "encoder_decoder":
        start = max(1, min(seq_len, int(seq_len * prefix_fraction))) - 1
        return hidden[:, start:], targets[:, start:]
    return hidden, targets


def optimizer_for(model, lr, weight_decay, device):
    groups = [{"params": [p for p in model.parameters() if p.ndim >= 2], "weight_decay": weight_decay},
              {"params": [p for p in model.parameters() if p.ndim < 2], "weight_decay": 0.0}]
    kwargs = {"lr": lr, "betas": (0.9, 0.95), "eps": 1e-8}
    if "fused" in inspect.signature(torch.optim.AdamW).parameters:
        kwargs["fused"] = device.type == "cuda"
    return torch.optim.AdamW(groups, **kwargs)


def amp_context(device, enabled):
    return torch.autocast(device_type=device.type, dtype=torch.float16, enabled=enabled and device.type == "cuda")


@torch.no_grad()
def validate(raw, dataset, cfg, device, world, rank):
    was_training = raw.training
    raw.eval()
    total = torch.zeros(2, device=device, dtype=torch.float64)
    micro = cfg["micro_batch"]
    batches = min(cfg["validation_batches"], len(dataset) // (world * micro))
    if batches == 0:
        raise ValueError("Validation cache is too small for this batch/world size.")
    for batch_idx in range(batches):
        first = (batch_idx * world + rank) * micro
        blocks = torch.stack([dataset[first + j] for j in range(micro)]).to(device, non_blocking=True)
        inputs, targets, source = make_batch(blocks, cfg["architecture"], cfg["prefix_fraction"], "continuation")
        with amp_context(device, cfg["fp16"]):
            hidden = raw(inputs, source_ids=source)
            hidden, targets = crop_loss_view(hidden, targets, cfg["architecture"], cfg["prefix_fraction"], "continuation", cfg["seq_len"])
            _, loss_sum, count = language_model_loss(hidden, targets, raw.embedding.weight, cfg["loss_chunk_tokens"])
        total[0] += loss_sum.double()
        total[1] += count.double()
    if world > 1:
        dist.all_reduce(total, op=dist.ReduceOp.SUM)
    raw.train(was_training)
    nll = (total[0] / total[1]).item()
    return {"validation_nll": nll, "validation_perplexity": math.exp(min(nll, 50)),
            "validation_target_tokens": int(total[1].item())}


def checkpoint_state(raw, optimizer, scaler, cfg, progress, best_nll, device, world, rank):
    state = rng_state(device)
    if world > 1:
        states = [None] * world
        dist.all_gather_object(states, state)
    else:
        states = [state]
    if rank == 0:
        destination = Path(cfg["output_dir"])
        payload = {"model": raw.state_dict(), "model_config": asdict(raw.config),
                   "optimizer": optimizer.state_dict(), "scaler": scaler.state_dict(),
                   "run_config": cfg, "progress": progress, "best_nll": best_nll,
                   "rng_states": states, "world_size": world,
                   "cache_fingerprint": cfg["cache_fingerprint"],
                   "tokenizer_sha256": cfg["tokenizer_sha256"],
                   "training_origin": "random_initialization_official_phase1_only"}
        temporary = destination / "latest.pt.tmp"
        torch.save(payload, temporary)
        temporary.replace(destination / "latest.pt")
    if world > 1:
        dist.barrier()


def save_best(raw, cfg, progress, metrics, rank):
    if rank != 0:
        return
    payload = {"model": {k: v.detach().cpu() for k, v in raw.state_dict().items()},
               "model_config": asdict(raw.config), "progress": progress,
               "metrics": metrics, "cache_fingerprint": cfg["cache_fingerprint"],
               "tokenizer_sha256": cfg["tokenizer_sha256"],
               "training_origin": "random_initialization_official_phase1_only"}
    destination = Path(cfg["output_dir"])
    torch.save(payload, destination / "best_model.pt.tmp")
    (destination / "best_model.pt.tmp").replace(destination / "best_model.pt")
    atomic_json(destination / "best_metrics.json", {**metrics, **progress, "parameters": parameter_count(raw)})


def learning_rate(tokens_seen, cfg):
    warm = min(cfg["warmup_tokens"], max(1, int(cfg["target_tokens"] * 0.05)))
    if tokens_seen < warm:
        return cfg["learning_rate"] * max(1, tokens_seen) / warm
    fraction = min(1.0, (tokens_seen - warm) / max(1, cfg["target_tokens"] - warm))
    return cfg["min_learning_rate"] + 0.5 * (cfg["learning_rate"] - cfg["min_learning_rate"]) * (1 + math.cos(math.pi * fraction))


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", required=True)
    ap.add_argument("--resume", default=None)
    ap.add_argument("--init-from", default=None)
    ap.add_argument("--benchmark", action="store_true")
    ap.add_argument("--benchmark-steps", type=int, default=30)
    args = ap.parse_args()
    cfg = json.loads(Path(args.config).read_text())
    device, world, rank = setup_distributed()
    torch.set_num_threads(max(1, cfg.get("cpu_threads", 2)))
    random.seed(cfg["seed"] + rank)
    torch.manual_seed(cfg["seed"])  # Identical initialization on every rank.
    if device.type == "cuda":
        torch.cuda.manual_seed_all(cfg["seed"])
    out = Path(cfg["output_dir"])
    out.mkdir(parents=True, exist_ok=True)
    cache = Path(cfg["cache_dir"])
    manifest = json.loads((cache / "manifest.json").read_text())
    assert manifest["complete"] and manifest["fingerprint"] == cfg["cache_fingerprint"]
    assert cfg["tokenizer_sha256"] == file_hash(cfg["tokenizer_path"])
    assert manifest["settings"]["tokenizer_sha256"] == cfg["tokenizer_sha256"]
    for split in ("train", "val"):
        assert (cache / f"{split}.bin").stat().st_size == manifest[split]["tokens"] * 2
    c = ModelConfig(**cfg["model_config"]) if "model_config" in cfg else architecture_config(
        cfg["architecture"], dropout=cfg["dropout"], gradient_checkpointing=cfg["gradient_checkpointing"])
    assert c.architecture == cfg["architecture"]
    raw = LegalLM(c).to(device)
    count = parameter_count(raw)
    if not cfg.get("allow_tiny_test_model", False):
        assert 50_000_000 <= count <= 300_000_000, f"Invalid parameter count: {count:,}"
    optimizer = optimizer_for(raw, cfg["learning_rate"], cfg["weight_decay"], device)
    try:
        scaler = torch.amp.GradScaler("cuda", enabled=cfg["fp16"] and device.type == "cuda")
    except (AttributeError, TypeError):
        scaler = torch.cuda.amp.GradScaler(enabled=cfg["fp16"] and device.type == "cuda")
    progress = {"step": 0, "tokens_seen": 0, "input_tokens_seen": 0, "epoch": 0, "batch_offset": 0}
    best_nll = float("inf")
    if args.resume or args.init_from:
        previous = torch.load(args.resume or args.init_from, map_location="cpu", weights_only=False)
        assert previous["training_origin"] == "random_initialization_official_phase1_only"
        assert previous["tokenizer_sha256"] == cfg["tokenizer_sha256"]
        assert previous["model_config"] == asdict(c)
        raw.load_state_dict(previous["model"])
        if args.resume:
            assert previous["cache_fingerprint"] == cfg["cache_fingerprint"]
            assert previous["world_size"] == world, "Exact resume requires the same GPU/process count."
            for key in ("seq_len", "micro_batch", "accumulation_steps", "objective", "prefix_fraction", "seed",
                        "target_tokens", "learning_rate", "min_learning_rate", "warmup_tokens", "weight_decay", "grad_clip"):
                assert previous["run_config"][key] == cfg[key], f"Resume mismatch: {key}. Use --init-from for a new schedule."
            optimizer.load_state_dict(previous["optimizer"])
            scaler.load_state_dict(previous["scaler"])
            progress, best_nll = previous["progress"], previous["best_nll"]
            saved_rng = previous["rng_states"][rank]
            if rank == 0 and not (out / "best_model.pt").exists() and math.isfinite(best_nll):
                previous_dir = Path(args.resume).parent
                if not (previous_dir / "best_model.pt").exists():
                    raise ValueError("Resume also requires the saved best_model.pt next to latest.pt; retain both files.")
                shutil.copyfile(previous_dir / "best_model.pt", out / "best_model.pt")
                if (previous_dir / "best_metrics.json").exists():
                    shutil.copyfile(previous_dir / "best_metrics.json", out / "best_metrics.json")
        else:
            saved_rng = None
        del previous
    else:
        saved_rng = None
    if world > 1:
        model = DDP(raw, device_ids=[device.index] if device.type == "cuda" else None,
                    broadcast_buffers=False, gradient_as_bucket_view=True, find_unused_parameters=False)
    else:
        model = raw
    # Same initialized model, independent dropout RNG stream per rank.
    # Exact resume restores the saved rank-specific state below.
    torch.manual_seed(cfg["seed"] + rank)
    if device.type == "cuda":
        torch.cuda.manual_seed(cfg["seed"] + rank)
    train = TokenBlocks(cache / "train.bin", cfg["seq_len"])
    val = TokenBlocks(cache / "val.bin", cfg["seq_len"])
    sampler = DistributedSampler(train, num_replicas=world, rank=rank, shuffle=True,
                                 seed=cfg["seed"], drop_last=True)
    loader_args = dict(batch_size=cfg["micro_batch"], sampler=sampler, drop_last=True,
                       num_workers=cfg["num_workers"], pin_memory=device.type == "cuda")
    if cfg["num_workers"]:
        # NCCL is not fork-safe. Workers read CPU token mappings only, using spawn.
        loader_args.update(persistent_workers=True, prefetch_factor=2, multiprocessing_context="spawn")
    loader = DataLoader(train, **loader_args)
    if len(loader) == 0:
        raise ValueError("Training cache too small for this batch/world size.")
    sampler.set_epoch(progress["epoch"])
    iterator = iter(loader)
    for _ in range(progress["batch_offset"]):
        next(iterator)
    if saved_rng is not None:
        restore_rng(saved_rng, device)
    stop_requested = [False]
    def request_stop(signum, frame):
        stop_requested[0] = True
    signal.signal(signal.SIGTERM, request_stop)
    # torchrun/kernel interruption can terminate workers before a new save;
    # latest.pt is guaranteed only at completed save points.
    if rank == 0:
        atomic_json(out / "run_config.json", cfg)
        atomic_json(out / "environment.json", {"torch": torch.__version__, "world_size": world,
                    "device": torch.cuda.get_device_name(device) if device.type == "cuda" else "CPU",
                    "cuda": torch.version.cuda, "parameters": count,
                    "capability": torch.cuda.get_device_capability(device) if device.type == "cuda" else None})
        print(f"{cfg['architecture']}: {count:,} parameters; {world} process(es); device={device}", flush=True)
    model.train()
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(device)
    synchronize(device)
    start_time = time.perf_counter()
    benchmark_start, benchmark_start_tokens = None, 0
    initial_tokens = progress["tokens_seen"]
    log_time, log_tokens = start_time, initial_tokens
    last_metrics, skips = {}, 0
    while progress["tokens_seen"] < cfg["target_tokens"] or args.benchmark:
        optimizer.zero_grad(set_to_none=True)
        step_stats = torch.zeros(2, device=device, dtype=torch.float64)
        step_target_tokens, step_input_tokens = 0, 0
        lr = learning_rate(progress["tokens_seen"] + 1, cfg)
        for group in optimizer.param_groups:
            group["lr"] = lr
        for micro_idx in range(cfg["accumulation_steps"]):
            try:
                blocks = next(iterator)
            except StopIteration:
                progress["epoch"] += 1
                progress["batch_offset"] = 0
                sampler.set_epoch(progress["epoch"])
                iterator = iter(loader)
                blocks = next(iterator)
            progress["batch_offset"] += 1
            blocks = blocks.to(device, non_blocking=True)
            inputs, targets, source = make_batch(blocks, cfg["architecture"], cfg["prefix_fraction"], cfg["objective"])
            sync_context = model.no_sync() if world > 1 and micro_idx < cfg["accumulation_steps"] - 1 else contextlib.nullcontext()
            with sync_context:  # Both forward AND backward belong inside no_sync.
                with amp_context(device, cfg["fp16"]):
                    hidden = model(inputs, source_ids=source)
                    hidden, targets = crop_loss_view(hidden, targets, cfg["architecture"], cfg["prefix_fraction"], cfg["objective"], cfg["seq_len"])
                    loss, loss_sum, supervised = language_model_loss(
                        hidden, targets, raw.embedding.weight, cfg["loss_chunk_tokens"], cfg["checkpoint_loss_chunks"])
                scaler.scale(loss / cfg["accumulation_steps"]).backward()
            step_stats[0] += loss_sum.double()
            step_stats[1] += supervised.double()
            step_target_tokens += targets.numel()
            step_input_tokens += inputs.numel() + (source.numel() if source is not None else 0)
        scaler.unscale_(optimizer)
        norm = torch.nn.utils.clip_grad_norm_(raw.parameters(), cfg["grad_clip"])
        old_scale = scaler.get_scale()
        scaler.step(optimizer)
        scaler.update()
        skipped = scaler.get_scale() < old_scale
        progress["input_tokens_seen"] += step_input_tokens * world
        if skipped:
            skips += 1
            if skips > 50:
                raise RuntimeError("More than 50 AMP-overflow updates. Reduce LR; inspect data and gradients.")
            continue
        progress["step"] += 1
        progress["tokens_seen"] += step_target_tokens * world
        step = progress["step"]
        if args.benchmark:
            warmup = min(10, max(1, args.benchmark_steps // 3))
            if step == warmup:
                synchronize(device)
                benchmark_start, benchmark_start_tokens = time.perf_counter(), progress["tokens_seen"]
            if step >= args.benchmark_steps:
                synchronize(device)
                elapsed = time.perf_counter() - benchmark_start
                peak = torch.cuda.max_memory_allocated(device) / 2**30 if device.type == "cuda" else 0.0
                reserved = torch.cuda.max_memory_reserved(device) / 2**30 if device.type == "cuda" else 0.0
                summary = torch.tensor([elapsed, peak, reserved], device=device, dtype=torch.float64)
                if world > 1:
                    dist.all_reduce(summary, op=dist.ReduceOp.MAX)
                measured_tokens = progress["tokens_seen"] - benchmark_start_tokens
                result = {"loss_target_tokens_per_second": measured_tokens / summary[0].item(),
                          "peak_allocated_gib_per_gpu": summary[1].item(), "warmup_steps": warmup,
                          "peak_reserved_gib_per_gpu": summary[2].item(),
                          "measured_steps": step - warmup, "parameters": count, "world_size": world,
                          "seq_len": cfg["seq_len"], "micro_batch": cfg["micro_batch"],
                          "accumulation_steps": cfg["accumulation_steps"], "architecture": cfg["architecture"]}
                if rank == 0:
                    atomic_json(out / "benchmark.json", result)
                    print(json.dumps(result, indent=2), flush=True)
                break
            continue
        if step % cfg["log_every"] == 0 or step == 1:
            if world > 1:
                dist.all_reduce(step_stats, op=dist.ReduceOp.SUM)
            synchronize(device)
            now = time.perf_counter()
            if rank == 0:
                row = {"step": step, "tokens_seen": progress["tokens_seen"], "train_nll": (step_stats[0] / step_stats[1]).item(),
                       "learning_rate": lr, "gradient_norm": float(norm),
                       "tokens_per_second": (progress["tokens_seen"] - log_tokens) / max(1e-6, now - log_time),
                       "elapsed_hours": (now - start_time) / 3600, "amp_skipped_updates": skips}
                log_path = out / "train_log.csv"
                fresh = not log_path.exists()
                with open(log_path, "a", newline="") as f:
                    writer = csv.DictWriter(f, fieldnames=list(row))
                    if fresh:
                        writer.writeheader()
                    writer.writerow(row)
                print(json.dumps(row), flush=True)
            log_time, log_tokens = now, progress["tokens_seen"]
        should_stop = False
        if step % cfg.get("time_check_every", 10) == 0 or progress["tokens_seen"] >= cfg["target_tokens"]:
            flag = torch.tensor(int(stop_requested[0] or time.perf_counter() - start_time >= cfg["max_hours"] * 3600), device=device)
            if world > 1:
                dist.all_reduce(flag, op=dist.ReduceOp.MAX)
            should_stop = bool(flag.item())
        final = should_stop or progress["tokens_seen"] >= cfg["target_tokens"]
        evaluate = step % cfg["eval_every"] == 0 or final
        if evaluate:
            last_metrics = validate(raw, val, cfg, device, world, rank)
            if not math.isfinite(last_metrics["validation_nll"]):
                raise RuntimeError("Validation loss is not finite.")
            if last_metrics["validation_nll"] < best_nll:
                best_nll = last_metrics["validation_nll"]
                save_best(raw, cfg, progress, last_metrics, rank)
            if rank == 0:
                with open(out / "validation_log.jsonl", "a") as f:
                    f.write(json.dumps({**progress, **last_metrics}) + "\n")
                print("VALIDATION", json.dumps(last_metrics), flush=True)
        if evaluate or final or step % cfg["save_every"] == 0:
            checkpoint_state(raw, optimizer, scaler, cfg, progress, best_nll, device, world, rank)
        if final:
            break
    if not args.benchmark and rank == 0:
        atomic_json(out / "training_summary.json", {**progress, **last_metrics, "best_validation_nll": best_nll,
                    "parameters": count, "wall_hours_this_session": (time.perf_counter() - start_time) / 3600,
                    "stopped_before_target": progress["tokens_seen"] < cfg["target_tokens"],
                    "amp_skipped_updates": skips})
    if world > 1:
        dist.destroy_process_group()


if __name__ == "__main__":
    main()


In [ ]:
from train_phase1 import TokenBlocks
sample=TokenBlocks(CACHE_DIR/'train.bin',SEQ_LEN)[0]
assert sample.shape==(SEQ_LEN+1,) and sample.dtype==torch.long
assert 0<=sample.min() and sample.max()<32768
print('Training blocks:',len(TokenBlocks(CACHE_DIR/'train.bin',SEQ_LEN)))
print('Validation blocks:',len(TokenBlocks(CACHE_DIR/'val.bin',SEQ_LEN)))
print('Cached ID range in first block:',int(sample.min()),int(sample.max()))

def launch(arguments):
    env={**os.environ,'TOKENIZERS_PARALLELISM':'false','OMP_NUM_THREADS':'2'}
    process=subprocess.Popen([str(a) for a in arguments],cwd=PROJECT,env=env,
        stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,start_new_session=True)
    tail=[]
    try:
        for line in process.stdout:
            print(line,end='')
            tail.append(line)
            tail=tail[-100:]
        code=process.wait()
    except KeyboardInterrupt:
        os.killpg(process.pid,signal.SIGINT)
        try: process.wait(timeout=20)
        except subprocess.TimeoutExpired: os.killpg(process.pid,signal.SIGKILL)
        raise
    if code:
        raise RuntimeError('Subprocess failed:\n'+''.join(tail))

def torchrun_command(script,*args):
    import socket
    with socket.socket() as socket_for_port:
        socket_for_port.bind(('127.0.0.1',0))
        port=socket_for_port.getsockname()[1]
    return [sys.executable,'-m','torch.distributed.run','--master_addr=127.0.0.1',
            f'--master_port={port}',f'--nproc_per_node={WORLD_SIZE}',str(PROJECT/script),*map(str,args)]

def run_config(architecture,output_dir,target_tokens,max_hours,objective='full',
               micro_batch=None,accumulation_steps=None,seq_len=None):
    return dict(architecture=architecture,output_dir=str(output_dir),target_tokens=target_tokens,max_hours=max_hours,
        objective=objective,prefix_fraction=.80,seq_len=seq_len or SEQ_LEN,
        micro_batch=micro_batch or MICRO_BATCH,accumulation_steps=accumulation_steps or ACCUMULATION_STEPS,
        seed=SEED,fp16=True,num_workers=NUM_WORKERS,cpu_threads=2,dropout=DROPOUT,
        gradient_checkpointing=GRADIENT_CHECKPOINTING,checkpoint_loss_chunks=CHECKPOINT_LOSS_CHUNKS,
        loss_chunk_tokens=LOSS_CHUNK_TOKENS,learning_rate=LEARNING_RATE,min_learning_rate=MIN_LEARNING_RATE,
        warmup_tokens=WARMUP_TOKENS,weight_decay=WEIGHT_DECAY,grad_clip=1.0,validation_batches=VALIDATION_BATCHES,
        log_every=10,eval_every=100,save_every=100,time_check_every=10,
        cache_dir=str(CACHE_DIR),cache_fingerprint=cache_manifest['fingerprint'],
        tokenizer_path=str(TOKENIZER_PATH),tokenizer_sha256=TOKENIZER_SHA256)

def execute_training(cfg,resume=None,init_from=None,benchmark=False):
    output=Path(cfg['output_dir']);output.mkdir(parents=True,exist_ok=True)
    config_path=output/'launch_config.json';atomic_json(config_path,cfg)
    args=['--config',config_path]
    if resume: args+=['--resume',resume]
    if init_from: args+=['--init-from',init_from]
    if benchmark: args+=['--benchmark','--benchmark-steps',BENCHMARK_STEPS]
    launch(torchrun_command('train_phase1.py',*args))

def matching_completed_run(cfg):
    return prepare_foundation_run(cfg)[2]

## 11. Benchmark every actual foundation objective on both T4 GPUs

The encoder–decoder benchmark uses the conditional suffix objective; the other foundations use full next-token prediction. Each benchmark starts with disposable random weights. It reduces the micro-batch on a genuine OOM while preserving the effective batch through increased accumulation. No benchmark weight is used for submissions. Existing completed training is reused independently of whether a new benchmark is run.

The SDPA profiler cell identifies the actual kernel on your Kaggle image. T4 does not imply that the standard FlashAttention-2 CUDA backend is active. Measure throughput and memory instead of assuming a published speedup.

In [ ]:
from torch.profiler import profile,ProfilerActivity
from torch.nn import functional as F
q=torch.randn(2,8,128,64,device='cuda',dtype=torch.float16)
with profile(activities=[ProfilerActivity.CPU,ProfilerActivity.CUDA]) as prof:
    for _ in range(3):
        F.scaled_dot_product_attention(q,q,q,is_causal=True)
    torch.cuda.synchronize()
print('Observed attention operators:',[x.key for x in prof.key_averages() if 'attention' in x.key.lower()])
del q,prof
torch.cuda.empty_cache()

In [ ]:
BENCHMARKS={};GEOMETRY={}
def fit_benchmark(name,objective,label):
    micro,accum=MICRO_BATCH,ACCUMULATION_STEPS
    while True:
        cfg=run_config(name,PROJECT/'benchmarks'/label,MAIN_TARGET_TOKENS,.25,
                       objective=objective,micro_batch=micro,accumulation_steps=accum)
        try:
            execute_training(cfg,benchmark=True)
            measured=json.loads((Path(cfg['output_dir'])/'benchmark.json').read_text())
            if measured['peak_reserved_gib_per_gpu']>13.0 and micro>1:
                print('Reducing micro-batch to leave room for other CUDA allocations.')
                micro=max(1,micro//2);accum*=2
                continue
            return cfg,measured
        except RuntimeError as error:
            if 'out of memory' not in str(error).lower() or micro==1:
                raise
            print('CUDA OOM during preflight; retrying a smaller micro-batch.')
            micro=max(1,micro//2);accum*=2
FOUNDATION_BENCH_CONFIGS={};FOUNDATION_BENCHMARKS={}
for name in FOUNDATION_ARCHITECTURES:
    cfg,measured=fit_benchmark(name,FOUNDATION_OBJECTIVES[name],f'foundation_{name}')
    FOUNDATION_BENCH_CONFIGS[name]=cfg;FOUNDATION_BENCHMARKS[name]=measured
    if RUN_ARCHITECTURE_SWEEP:
        if FOUNDATION_OBJECTIVES[name]=='continuation':
            pilot_cfg,pilot_measured=cfg,measured
        else:
            pilot_cfg,pilot_measured=fit_benchmark(name,'continuation',f'pilot_{name}')
        BENCHMARKS[name]=pilot_measured
        GEOMETRY[name]=(pilot_cfg['micro_batch'],pilot_cfg['accumulation_steps'])
display(pd.DataFrame(FOUNDATION_BENCHMARKS.values()))
atomic_json(PROJECT/'benchmark_summary.json',{'pilots':BENCHMARKS,'foundations':FOUNDATION_BENCHMARKS})

## 12. Set a realistic time budget before the long run

For a measured rate $r$ and supervised-token target $D$, a useful lower estimate is $D/r$. The cell adds a 25% allowance for evaluation, checkpoints and variability, then applies each configured time cap. Data preparation and generation are separate costs. These allowances are estimates, not promised runtimes.

If the displayed plan leaves too little time for **at least five candidate submissions and competition scoring**, reduce individual foundation token targets before starting new runs. Keep the pilot target identical across architectures where possible. A pilot stopped early by its time cap is not an equal-token comparison and is marked as such in the table. The optional pilots are exploratory; assess architecture quality using the measured foundation runs and generation results.

The main 200M-token target corresponds to about four tokens/parameter. More official-data training is likely worth exploring if compute remains, but the large corpus is an available resource rather than a requirement to consume all 4.5B+ tokens. Record the tokens actually processed, not merely the requested token budget.

In [ ]:
timing=[]
if RUN_ARCHITECTURE_SWEEP:
    for name,b in BENCHMARKS.items():
        estimate=PILOT_TARGET_TOKENS/b['loss_target_tokens_per_second']/3600*1.25
        timing.append({'stage':f'pilot {name}','requested_supervised_tokens':PILOT_TARGET_TOKENS,
                       'estimated_hours_with_allowance':estimate,'cap_hours_per_invocation':PILOT_MAX_HOURS})
for name in FOUNDATION_ARCHITECTURES:
    b=FOUNDATION_BENCHMARKS[name]
    target=FOUNDATION_TARGET_TOKENS[name]
    bench=FOUNDATION_BENCH_CONFIGS[name]
    cfg=run_config(name,PROJECT/'runs'/f'main_{name}',target,FOUNDATION_MAX_HOURS[name],
        objective=FOUNDATION_OBJECTIVES[name],micro_batch=bench['micro_batch'],
        accumulation_steps=bench['accumulation_steps'])
    actual,resume,completed=prepare_foundation_run(cfg)
    summary_path=Path(actual['output_dir'])/'training_summary.json'
    previous=json.loads(summary_path.read_text()).get('tokens_seen',0) if summary_path.exists() else 0
    remaining=max(0,target-previous)
    estimate=remaining/b['loss_target_tokens_per_second']/3600*1.25
    timing.append({'stage':f'foundation {name}','objective':FOUNDATION_OBJECTIVES[name],
        'requested_supervised_tokens':target,'remaining_supervised_tokens':remaining,
        'estimated_hours_with_allowance':estimate,'cap_hours_per_invocation':FOUNDATION_MAX_HOURS[name],
        'compatible_run_completed':completed})
display(pd.DataFrame(timing))
print('Estimated capped training hours this invocation:',sum(min(r['estimated_hours_with_allowance'],r['cap_hours_per_invocation']) for r in timing))
print('Generation, preprocessing, checkpoint restore, downloads and Codabench scoring take additional time.')

## 13. Train the four architecture pilots

Each pilot starts randomly and uses the same tokenizer, cached sources, holdout, sequence length, prefix fraction and **supervised-token target**. Different matrix/attention/recurrent costs mean equal target tokens do not imply equal wall time. Compare both quality and resource use. Data blocks are deterministically shuffled, and source-token proportions come from the packed cache.

The automatic table contains measured NLL, perplexity, processed tokens, runtime and parameter count. It does not produce an analytical conclusion for your report. Inspect generation samples and learning curves as well: an undertrained pilot may rank differently after a larger budget. All four foundation architectures are trained in the next stage; `MAIN_ARCHITECTURE` selects the primary model for later phases.

In [ ]:
PILOT_RUNS={};experiment_rows=[]
if RUN_ARCHITECTURE_SWEEP:
    for name in ARCHITECTURES:
        micro,accum=GEOMETRY[name]
        cfg=run_config(name,PROJECT/'runs'/f'pilot_{name}',PILOT_TARGET_TOKENS,PILOT_MAX_HOURS,
                       objective='continuation',micro_batch=micro,accumulation_steps=accum)
        folder=Path(cfg['output_dir'])
        summary_path=folder/'training_summary.json'
        actual,resume,completed=prepare_foundation_run(cfg)
        if not completed:
            execute_training(actual,resume=resume)
        summary=json.loads(summary_path.read_text())
        best=json.loads((folder/'best_metrics.json').read_text())
        experiment_rows.append({'architecture':name,'parameters':best['parameters'],
            'best_common_suffix_nll':best['validation_nll'],'perplexity':best['validation_perplexity'],
            'actual_supervised_tokens':summary['tokens_seen'],'best_checkpoint_tokens':best['tokens_seen'],
            'wall_hours_last_session':summary['wall_hours_this_session'],
            'stopped_before_target':summary['stopped_before_target'],
            'pilot_objective':'common suffix'})
        PILOT_RUNS[name]=folder
    experiment_table=pd.DataFrame(experiment_rows).sort_values('best_common_suffix_nll')
    display(experiment_table)
    experiment_table.to_csv(PROJECT/'architecture_experiments.csv',index=False)
else:
    print('Optional pilots disabled. The next stage trains all four foundation architectures.')

## 14. Train or recover all four foundation runs

Each architecture has its own `runs/main_<architecture>` directory, optimizer, learning-rate schedule, checkpoints, logs, and best-validation weights. The default target is 200M supervised tokens for each. The encoder–decoder trains on the suffix conditional on an encoded prefix. The other three train on all next-token positions. Validation always uses the common suffix objective, so compare the same heldout target positions and also inspect generation quality.

A compatible completed run is retained. An incomplete run resumes `latest.pt` with its original optimizer, scaler, random states, data cursor, and batch geometry; restoring paths and extending the per-invocation wall-time allowance is supported. A changed tokenizer, cache, architecture, objective, target, or optimizer schedule is rejected. The notebook does not pretend that stopping at a time cap means reaching the token target.

Your already completed `main_decoder` run from the previous notebook has the same defaults and can be reused. Keep its `run_config.json`, `training_summary.json`, `best_metrics.json`, `best_model.pt`, and `latest.pt`, plus matching tokenizer/cache. The other models receive foundation training rather than being used only as 5M-token pilots.

In [ ]:
FOUNDATION_RUNS={};FOUNDATION_CHECKPOINTS={};FOUNDATION_CONFIGS={};foundation_rows=[]
for name in FOUNDATION_ARCHITECTURES:
    folder=PROJECT/'runs'/f'main_{name}'
    bench=FOUNDATION_BENCH_CONFIGS[name]
    expected=run_config(name,folder,FOUNDATION_TARGET_TOKENS[name],FOUNDATION_MAX_HOURS[name],
        objective=FOUNDATION_OBJECTIVES[name],micro_batch=bench['micro_batch'],
        accumulation_steps=bench['accumulation_steps'])
    actual,resume,completed=prepare_foundation_run(expected)
    if RUN_MAIN_TRAINING and not completed:
        init_from=None
        if FOUNDATION_INIT_FROM_PILOT and resume is None:
            assert name in PILOT_RUNS, 'Enable/run your own architecture pilot first.'
            init_from=PILOT_RUNS[name]/'best_model.pt'
            actual['parent_checkpoint_sha256']=file_hash(init_from)
            actual['parent_stage']='own same-architecture pilot; fresh foundation optimizer/schedule'
        execute_training(actual,resume=resume,init_from=init_from)
    elif completed:
        print('Reusing completed foundation:',name)
    checkpoint=folder/'best_model.pt'
    assert checkpoint.exists(), f'{name}: train or restore your own foundation best_model.pt.'
    summary=json.loads((folder/'training_summary.json').read_text())
    best=json.loads((folder/'best_metrics.json').read_text())
    FOUNDATION_RUNS[name]=folder;FOUNDATION_CHECKPOINTS[name]=checkpoint;FOUNDATION_CONFIGS[name]=actual
    foundation_rows.append({'architecture':name,'parameters':best['parameters'],
        'training_objective':FOUNDATION_OBJECTIVES[name],'target_supervised_tokens':FOUNDATION_TARGET_TOKENS[name],
        'actual_supervised_tokens':summary['tokens_seen'],'input_tokens_seen':summary['input_tokens_seen'],
        'best_checkpoint_tokens':best['tokens_seen'],'best_common_suffix_nll':best['validation_nll'],
        'common_suffix_perplexity':best['validation_perplexity'],
        'wall_hours_last_invocation':summary['wall_hours_this_session'],
        'target_reached':summary['tokens_seen']>=FOUNDATION_TARGET_TOKENS[name],
        'checkpoint':str(checkpoint),'checkpoint_sha256':file_hash(checkpoint)})
    print('Foundation ready:',name,'| supervised tokens:',summary['tokens_seen'])
foundation_table=pd.DataFrame(foundation_rows)
display(foundation_table.sort_values('best_common_suffix_nll'))
foundation_table.to_csv(PROJECT/'foundation_experiments.csv',index=False)
atomic_json(PROJECT/'foundation_manifest.json',{'models':foundation_rows,'dataset_revision':REVISION,
    'tokenizer_sha256':TOKENIZER_SHA256,'cache_fingerprint':cache_manifest['fingerprint']})
MAIN_RUN=FOUNDATION_RUNS[MAIN_ARCHITECTURE]
FINAL_CHECKPOINT=FOUNDATION_CHECKPOINTS[MAIN_ARCHITECTURE]
CHECKPOINT_HASHES={row['architecture']:row['checkpoint_sha256'] for row in foundation_rows}

## 15. Inspect training evidence

Use curves to identify unstable learning rates, overfitting, undertraining and throughput changes. A lower conditional validation NLL can still give worse semantic continuations; the competition evaluates generation against a private reference. Plot actual results and retain your observations.

Training NLL here is the last logged update, not a smoothed epoch average. Validation is a fixed heldout subset. For a final checkpoint you can enlarge `validation_batches` and run a read-only validation pass using the helper without retraining; make the larger protocol explicit when comparing its NLL to previous numbers.

In [ ]:
import matplotlib.pyplot as plt
fig,axes=plt.subplots(len(FOUNDATION_ARCHITECTURES),3,figsize=(16,4*len(FOUNDATION_ARCHITECTURES)),squeeze=False)
for row,name in enumerate(FOUNDATION_ARCHITECTURES):
    folder=FOUNDATION_RUNS[name]
    train_log=pd.read_csv(folder/'train_log.csv')
    val_log=pd.read_json(folder/'validation_log.jsonl',lines=True)
    axes[row,0].plot(train_log.tokens_seen/1e6,train_log.train_nll,label='Training objective')
    axes[row,0].plot(val_log.tokens_seen/1e6,val_log.validation_nll,'o-',label='Common suffix validation')
    axes[row,0].set(title=name,xlabel='Supervised tokens (millions)',ylabel='NLL (nats)');axes[row,0].legend()
    axes[row,1].plot(train_log.tokens_seen/1e6,train_log.learning_rate)
    axes[row,1].set(xlabel='Supervised tokens (millions)',ylabel='Learning rate')
    axes[row,2].plot(train_log.tokens_seen/1e6,train_log.tokens_per_second)
    axes[row,2].set(xlabel='Supervised tokens (millions)',ylabel='Loss-target tokens/s')
fig.suptitle('Actual measurements: four foundation runs')
fig.tight_layout();fig.savefig(PROJECT/'training_curves_all_foundations.png',dpi=160);plt.show()

## 16. Resume across Kaggle sessions

To resume, retain the **entire relevant run folder** containing `latest.pt`, `best_model.pt`, configs and logs, along with `tokenizer.json` and the matching token-cache directory/manifest. Restore those files into this project's corresponding paths before rerunning. The `latest.pt` file alone can recover training state but the earlier best weights must also be available to preserve checkpoint selection.

`latest.pt` includes optimizer moments, GradScaler, RNG state per rank, sampler epoch and batch cursor. Resume with the same GPU/process count and batch geometry. The cache fingerprint and tokenizer SHA are checked. An exact resume is relative to the last **completed checkpoint**; work since that save may be lost if Kaggle terminates the process. The normal configured time stop completes an update, validates and saves before exiting. An abrupt kernel/session interruption can happen before a final save.

The training launcher stores trusted self-produced checkpoints using PyTorch serialization. Use the weights-only inference file for later-phase adaptation if you do not need optimizer recovery. A new tokenizer cannot be substituted while loading old token embeddings.

In [ ]:
# Disabled because the main loop already resumes matching incomplete runs.
RUN_RECOVERY_EXAMPLE=False
RECOVERY_ARCHITECTURE='encoder_decoder'
RECOVERY_CHECKPOINT=None
if RUN_RECOVERY_EXAMPLE:
    assert RECOVERY_CHECKPOINT
    saved=torch.load(RECOVERY_CHECKPOINT,map_location='cpu',weights_only=False)
    recovery_cfg=dict(saved['run_config'])
    assert recovery_cfg['architecture']==RECOVERY_ARCHITECTURE
    recovery_cfg.update(cache_dir=str(CACHE_DIR),tokenizer_path=str(TOKENIZER_PATH),
        output_dir=str(FOUNDATION_RUNS[RECOVERY_ARCHITECTURE]),max_hours=FOUNDATION_MAX_HOURS[RECOVERY_ARCHITECTURE])
    del saved
    execute_training(recovery_cfg,resume=RECOVERY_CHECKPOINT)

## 17. Optional 2048-token context experiment

If long prompt performance matters and time remains, continue **your own** main weights at 2048 tokens using a fresh lower-LR optimizer and a smaller micro-batch. This is a deliberate training-stage change, not an exact resume. Run its benchmark first, retain the original checkpoint, and compare generation on heldout 512-word prompts before deciding whether to use the new stage.

The default block length is not silently increased halfway through the main schedule. Changing context changes compute, effective token batch and validation task difficulty. The cell is disabled by default to protect the main training/generation budget. It does not automatically replace the selected checkpoint based on incomparable NLL values.

In [ ]:
RUN_LONG_CONTEXT_POLISH=False
if RUN_LONG_CONTEXT_POLISH:
    long_cfg=run_config(MAIN_ARCHITECTURE,PROJECT/'runs'/'long_context',20_000_000,1.0,
        objective=FOUNDATION_OBJECTIVES[MAIN_ARCHITECTURE],micro_batch=2,accumulation_steps=8,seq_len=2048)
    long_cfg.update(learning_rate=1e-4,min_learning_rate=1e-5,warmup_tokens=200_000)
    long_bench={**long_cfg,'output_dir':str(PROJECT/'benchmarks'/'long_context')}
    execute_training(long_bench,benchmark=True)
    long_cfg['parent_checkpoint_sha256']=file_hash(FINAL_CHECKPOINT)
    long_cfg['parent_stage']='own main Phase-1 model; 2048-token context extension'
    execute_training(long_cfg,init_from=FINAL_CHECKPOINT)
    print('Candidate long-context weights:',Path(long_cfg['output_dir'])/'best_model.pt')
    print('Compare heldout generation before manually updating FINAL_CHECKPOINT.')

## 18. Cached inference with token-cap recovery

Each GPU runs an independent copy of the selected architecture and processes a disjoint prompt shard. The decoder models cache self-attention keys/values; encoder–decoder caches encoded memory and cross-attention projections; the hybrid additionally carries recurrent state. Generation retains every official prompt token and outputs exactly 128 whitespace-separated continuation words.

The generator includes the previously tested **length-recovery patch**. It first uses the original 1024-token settings. If a prompt exhausts that cap, it retries with up to 2048 tokens within the 4096-position allowance. A further retry adds token n-gram blocking and a word-progress constraint to escape long repetitive fragments. Every fallback's actual settings are recorded in the per-ID audit. No filler or copied reference continuation is inserted. Inspect these cases for quality.

The original base decoding configuration is unchanged, so matching saved decoder partials can be reused. Completed predictions are saved before a subsequent inference error. Source cleanup calls `destroy_process_group()` on normal Python exits. Outputs must still pass the independent exact-ID, word-count, schema, and archive checks.

In [ ]:
%%writefile /kaggle/working/capstone_phase1/generate_phase1.py
"""Cached 128-word continuations, one inference worker/GPU, strict Codabench archive."""
import argparse
import hashlib
import json
import os
from pathlib import Path
import time
import zipfile
import torch
import torch.distributed as dist
from tokenizers import Tokenizer
from legal_models import LegalLM, ModelConfig, parameter_count
from legal_data import atomic_json, canonical_hash, file_hash
from train_phase1 import setup_distributed, amp_context


def sample_token(logits, generated, cfg, tokenizer, generator):
    logits = logits.float().clone()
    # The continuation must reach 128 complete words; EOS is disabled while
    # generating it. Other control tokens must never appear in submitted text.
    for spelling in ("<pad>", "<bos>", "<eos>", "<unk>"):
        idx = tokenizer.token_to_id(spelling)
        if idx is not None:
            logits[idx] = -float("inf")
    penalty = cfg.get("repetition_penalty", 1.0)
    if generated and penalty != 1.0:
        ids = torch.tensor(sorted(set(generated[-256:])), device=logits.device)
        values = logits[ids]
        logits[ids] = torch.where(values < 0, values * penalty, values / penalty)
    temperature = cfg.get("temperature", 0.8)
    if temperature == 0:
        return int(logits.argmax().item())
    logits = logits / temperature
    k = cfg.get("top_k", 50)
    if 0 < k < logits.numel():
        threshold = torch.topk(logits, k).values[-1]
        logits.masked_fill_(logits < threshold, -float("inf"))
    p = cfg.get("top_p", 0.95)
    if 0 < p < 1:
        values, indices = logits.sort(descending=True)
        remove = values.softmax(-1).cumsum(-1) > p
        remove[1:] = remove[:-1].clone()
        remove[0] = False
        values.masked_fill_(remove, -float("inf"))
        logits.scatter_(0, indices, values)
    return int(torch.multinomial(logits.softmax(-1), 1, generator=generator).item())


@torch.inference_mode()
def generate_continuation(model, tokenizer, prompt, cfg, device, example_id="0"):
    target_words = cfg.get("target_words", 128)
    # A trailing space puts generation at a word boundary, rather than allowing
    # continuation of the prompt's final word to count as a new word.
    tokens = tokenizer.encode(prompt.rstrip() + " ").ids
    if not tokens:
        raise ValueError(f"Empty prompt: {example_id}")
    if len(tokens) > cfg["prompt_max_tokens"]:
        raise ValueError(f"Prompt {example_id} has {len(tokens)} tokens; raise prompt_max_tokens. Full prompts are preserved.")
    if len(tokens) + cfg["max_new_tokens"] > model.config.max_positions and model.config.architecture != "encoder_decoder":
        raise ValueError("Prompt plus generation token cap exceeds the RoPE table. Increase max_positions before loading.")
    seed = cfg["seed"] + int(hashlib.sha256(str(example_id).encode()).hexdigest()[:8], 16)
    generator = torch.Generator(device=device).manual_seed(seed % (2**63 - 1))
    prompt_ids = torch.tensor([tokens], device=device, dtype=torch.long)
    if model.config.architecture == "encoder_decoder":
        current, source = prompt_ids[:, -1:], prompt_ids
    else:
        current, source = prompt_ids, None
    state, generated = None, []
    for index in range(cfg["max_new_tokens"]):
        with amp_context(device, cfg.get("fp16", True)):
            hidden, state = model(current, source_ids=source, state=state, use_cache=True)
            logits = model.project(hidden[:, -1:])[0, 0]
        next_id = sample_token(logits, generated, cfg, tokenizer, generator)
        generated.append(next_id)
        current = torch.tensor([[next_id]], device=device)
        source = None
        # Seeing a 129th word proves that the 128th word is complete, even if
        # the tokenizer emits several pieces per word. Tokens != words.
        if (index + 1) % 8 == 0 or index + 1 == cfg["max_new_tokens"]:
            decoded = tokenizer.decode(generated, skip_special_tokens=True)
            words = decoded.split()
            if len(words) >= target_words + 1:
                text = " ".join(words[:target_words])
                assert len(text.split()) == target_words
                return text, {"prompt_tokens": len(tokens), "generated_tokens": len(generated)}
    raise RuntimeError(f"{example_id}: token cap reached before {target_words} complete words. Increase max_new_tokens; no filler text is inserted.")


def validate_predictions(predictions, prompts, target_words=128):
    if not isinstance(predictions, dict) or set(predictions) != set(prompts):
        missing, extra = set(prompts) - set(predictions), set(predictions) - set(prompts)
        raise ValueError(f"ID mismatch: {len(missing)} missing, {len(extra)} extra.")
    for key, value in predictions.items():
        if not isinstance(value, dict) or set(value) != {"text"} or not isinstance(value["text"], str):
            raise ValueError(f"Invalid prediction schema for {key}.")
        if len(value["text"].split()) != target_words:
            raise ValueError(f"{key} must have exactly {target_words} whitespace-separated words.")
        if any(control in value["text"] for control in ("<pad>", "<bos>", "<eos>", "<unk>")):
            raise ValueError(f"Control token in {key}.")
    return True


def write_submission(predictions, prompts, output_dir, target_words=128):
    validate_predictions(predictions, prompts, target_words)
    root = Path(output_dir)
    root.mkdir(parents=True, exist_ok=True)
    atomic_json(root / "prediction.json", predictions)
    archive = root / "submission.zip"
    with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as z:
        # Stable zip metadata prevents timestamps from changing identical archives.
        info = zipfile.ZipInfo("prediction.json", date_time=(1980, 1, 1, 0, 0, 0))
        info.compress_type = zipfile.ZIP_DEFLATED
        z.writestr(info, (root / "prediction.json").read_bytes())
    with zipfile.ZipFile(archive) as z:
        assert z.namelist() == ["prediction.json"]
        recovered = json.loads(z.read("prediction.json").decode("utf-8"))
        validate_predictions(recovered, prompts, target_words)
    return archive


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--checkpoint", required=True)
    ap.add_argument("--tokenizer", required=True)
    ap.add_argument("--prompts", required=True)
    ap.add_argument("--config", required=True)
    ap.add_argument("--output-dir", required=True)
    args = ap.parse_args()
    device, world, rank = setup_distributed()
    torch.set_num_threads(2)
    cfg = json.loads(Path(args.config).read_text())
    if cfg.get("target_words", 128) != 128 and not cfg.get("allow_short_test", False):
        raise ValueError("Official Phase-1 submissions require 128 words.")
    prompts = json.loads(Path(args.prompts).read_text())
    assert prompts and all(isinstance(v, dict) and isinstance(v.get("text"), str) for v in prompts.values())
    tokenizer = Tokenizer.from_file(args.tokenizer)
    saved = torch.load(args.checkpoint, map_location="cpu", weights_only=False)
    assert saved["training_origin"] == "random_initialization_official_phase1_only"
    assert saved["tokenizer_sha256"] == file_hash(args.tokenizer)
    model = LegalLM(ModelConfig(**saved["model_config"]))
    model.load_state_dict(saved["model"])
    del saved
    model = model.to(device).eval()
    if device.type == "cuda" and cfg.get("fp16", True):
        model.half()
    if not cfg.get("allow_short_test", False):
        assert 50_000_000 <= parameter_count(model) <= 300_000_000
    root = Path(args.output_dir)
    root.mkdir(parents=True, exist_ok=True)
    run_id = canonical_hash({"checkpoint": file_hash(args.checkpoint), "tokenizer": file_hash(args.tokenizer),
                             "prompts": file_hash(args.prompts), "decode": cfg})
    shard_path = root / f"partial_{rank:02}.json"
    metadata_path = root / f"partial_{rank:02}.metadata.json"
    predictions, audit = {}, {}
    if metadata_path.exists() and shard_path.exists():
        existing = json.loads(metadata_path.read_text())
        if existing["run_id"] == run_id:
            predictions = json.loads(shard_path.read_text())
            audit = existing.get("audit", {})
    ids = list(prompts)[rank::world]
    # When process count changes, retain only this worker's current ID subset.
    predictions = {k: v for k, v in predictions.items() if k in set(ids)}
    start = time.perf_counter()
    for position, example_id in enumerate(ids):
        if example_id in predictions:
            continue
        try:
            text, details = generate_continuation(model, tokenizer, prompts[example_id]["text"], cfg, device, example_id)
        except Exception as error:
            atomic_json(shard_path, predictions)
            atomic_json(metadata_path, {"run_id": run_id, "audit": audit})
            atomic_json(root / f"failure_{rank:02}.json", {"example_id": example_id, "error": str(error)})
            raise
        predictions[example_id] = {"text": text}
        audit[example_id] = details
        if (position + 1) % cfg.get("save_every_examples", 10) == 0 or position + 1 == len(ids):
            atomic_json(shard_path, predictions)
            atomic_json(metadata_path, {"run_id": run_id, "audit": audit})
            print(f"rank {rank}: {position + 1}/{len(ids)} prompts", flush=True)
    # Also writes empty shards if a rank has no examples.
    atomic_json(shard_path, predictions)
    atomic_json(metadata_path, {"run_id": run_id, "audit": audit})
    if world > 1:
        dist.barrier()
    if rank == 0:
        merged, complete_audit = {}, {}
        for worker in range(world):
            shard = json.loads((root / f"partial_{worker:02}.json").read_text())
            meta = json.loads((root / f"partial_{worker:02}.metadata.json").read_text())
            assert meta["run_id"] == run_id
            if set(merged) & set(shard):
                raise ValueError("Duplicate IDs across generation shards.")
            merged.update(shard)
            complete_audit.update(meta["audit"])
        archive = write_submission(merged, prompts, root, cfg.get("target_words", 128))
        atomic_json(root / "generation_metadata.json", {
            "run_id": run_id, "decode": cfg, "examples": len(prompts),
            "seconds_this_session": time.perf_counter() - start, "world_size": world,
            "parameters": parameter_count(model), "archive_sha256": file_hash(archive),
            "audit": complete_audit,
        })
        print(f"Validated archive: {archive}", flush=True)
    if world > 1:
        dist.destroy_process_group()




# PHASE1_LENGTH_RECOVERY_V1
_original_sample_token = sample_token
_original_generate_continuation = generate_continuation
_word_start_cache = {}


def _word_start_ids(tokenizer):
    key = id(tokenizer)
    if key not in _word_start_cache:
        starts = set()
        for idx in tokenizer.get_vocab().values():
            text = tokenizer.decode([idx], skip_special_tokens=True)
            if text and text[0].isspace() and text.strip():
                starts.add(idx)
        _word_start_cache[key] = starts
    return _word_start_cache[key]


def sample_token(logits, generated, cfg, tokenizer, generator):
    n = int(cfg.get("no_repeat_ngram_size", 0))
    guard = int(cfg.get("word_progress_guard_tokens", 0))
    if not n and not guard:
        return _original_sample_token(logits, generated, cfg, tokenizer, generator)
    scores = logits.float().clone()
    if n > 1 and len(generated) >= n - 1:
        prefix = tuple(generated[-(n - 1):])
        banned = {generated[i + n - 1] for i in range(len(generated) - n + 1)
                  if tuple(generated[i:i + n - 1]) == prefix}
        if banned:
            scores[list(banned)] = -float("inf")
    if guard and len(generated) >= guard:
        current_words = len(tokenizer.decode(generated, skip_special_tokens=True).split())
        earlier_words = len(tokenizer.decode(generated[:-guard], skip_special_tokens=True).split())
        if current_words <= earlier_words:
            starts = _word_start_ids(tokenizer)
            if not starts:
                raise RuntimeError("Tokenizer has no whitespace-prefixed lexical tokens for length recovery.")
            allowed = torch.zeros_like(scores, dtype=torch.bool)
            allowed[list(starts)] = True
            scores.masked_fill_(~allowed, -float("inf"))
    if not torch.isfinite(scores).any():
        raise RuntimeError("No finite token scores remain under the recovery constraints.")
    return _original_sample_token(scores, generated, cfg, tokenizer, generator)


def generate_continuation(model, tokenizer, prompt, cfg, device, example_id="0"):
    prompt_tokens = len(tokenizer.encode(prompt.rstrip() + " ").ids)
    room = model.config.max_positions - (1 if model.config.architecture == "encoder_decoder" else prompt_tokens)
    extended_cap = min(room, max(int(cfg["max_new_tokens"]), 2048))
    attempts = [dict(cfg)]
    if extended_cap > int(cfg["max_new_tokens"]):
        attempts.append({**cfg, "max_new_tokens": extended_cap})
    attempts.append({**cfg, "max_new_tokens": extended_cap,
        "repetition_penalty": max(1.15, float(cfg.get("repetition_penalty", 1.0))),
        "no_repeat_ngram_size": 4, "word_progress_guard_tokens": 12,
        "seed": int(cfg["seed"]) + 1_000_003})
    failures = []
    for attempt, actual_cfg in enumerate(attempts):
        try:
            text, audit = _original_generate_continuation(
                model, tokenizer, prompt, actual_cfg, device, example_id)
        except RuntimeError as error:
            if "token cap reached before" not in str(error):
                raise
            failures.append({"attempt": attempt, "max_new_tokens": actual_cfg["max_new_tokens"],
                             "error": str(error)})
            print(f"rank {os.environ.get('RANK', '0')}: length recovery for ID {example_id}, "
                  f"attempt {attempt + 1}", flush=True)
            continue
        audit.update(recovery_policy="length-recovery-v1", recovery_attempt=attempt,
                     effective_decode=actual_cfg, previous_length_failures=failures)
        return text, audit
    raise RuntimeError(f"{example_id}: could not produce {cfg.get('target_words', 128)} complete words "
                       f"after {len(attempts)} bounded attempts. No filler text was inserted.")

if __name__ == "__main__":
    try:
        main()
    finally:
        if dist.is_initialized():
            dist.destroy_process_group()

In [ ]:
from generate_phase1 import validate_predictions,write_submission
DECODE_VARIANTS={
    'greedy':dict(temperature=0.0,top_p=1.0,top_k=0,repetition_penalty=1.0),
    'focused':dict(temperature=0.65,top_p=0.90,top_k=40,repetition_penalty=1.04),
    'balanced':dict(temperature=0.80,top_p=0.95,top_k=50,repetition_penalty=1.04),
    'diverse':dict(temperature=0.95,top_p=0.97,top_k=80,repetition_penalty=1.04),
}
CANDIDATE_PLAN=submission_plan()
PRIMARY_CANDIDATE='decoder_balanced'
def decode_config(variant,seed_offset=0):
    return {**DECODE_VARIANTS[variant],'seed':SEED+seed_offset,'fp16':True,'target_words':128,
            'max_new_tokens':1024,'prompt_max_tokens':2048,'save_every_examples':10}
def candidate_output(plan,heldout=False):
    if plan['architecture']=='decoder' and plan['seed_offset']==0:
        suffix=plan['variant']  # Preserve legacy decoder directories and their resume fingerprints.
    else: suffix=plan['label']
    return PROJECT/('heldout_generation' if heldout else 'submissions')/suffix
def execute_generation(variant,prompts_path,output_dir,checkpoint,seed_offset=0):
    config=decode_config(variant,seed_offset)
    output_dir=Path(output_dir);output_dir.mkdir(parents=True,exist_ok=True)
    prompts=json.loads(Path(prompts_path).read_text())
    expected_id=canonical_hash({'checkpoint':file_hash(checkpoint),'tokenizer':TOKENIZER_SHA256,
                              'prompts':file_hash(prompts_path),'decode':config})
    metadata_path=output_dir/'generation_metadata.json'
    predictions_path=output_dir/'prediction.json'
    if metadata_path.exists() and predictions_path.exists():
        metadata=json.loads(metadata_path.read_text())
        if metadata.get('run_id')==expected_id:
            predictions=json.loads(predictions_path.read_text())
            validate_predictions(predictions,prompts)
            write_submission(predictions,prompts,output_dir)
            print('Reusing complete generation:',output_dir.name)
            return predictions
    path=output_dir/'decode_config.json';atomic_json(path,config)
    launch(torchrun_command('generate_phase1.py','--checkpoint',checkpoint,'--tokenizer',TOKENIZER_PATH,
        '--prompts',prompts_path,'--config',path,'--output-dir',output_dir))
    predictions=json.loads(predictions_path.read_text())
    validate_predictions(predictions,prompts)
    return predictions
display(pd.DataFrame(CANDIDATE_PLAN[:5]))

## 19. Compare all five planned experiments on heldout continuations

Each planned model/decoding experiment generates the same balanced heldout prompt subset. References come only from heldout training-corpus documents; private official test references are unavailable. ROUGE-L and SacreBLEU are diagnostics; the official primary metric remains Codabench BERTScore F1. Inspect generated text, names/citations, and fallback audits, not just NLL.

The five initial experiments are decoder/balanced, decoder/focused, encoder–decoder/balanced, local GQA/balanced, and hybrid/balanced. Their trained weights are your own separate foundation checkpoints. Seeded sampling is deterministic per ID and independent of rank, supporting partial recovery. `RUN_HELDOUT_GENERATION=False` skips this diagnostic stage without reducing the five official candidate requirement.

In [ ]:
VALIDATION_GENERATION_LIMIT=20
groups={s:[p for p in VALIDATION_PAIRS if p['source']==s] for s in SOURCE_FILES}
selected=[];offset=0
while len(selected)<min(VALIDATION_GENERATION_LIMIT,len(VALIDATION_PAIRS)):
    added=False
    for source in SOURCE_FILES:
        if offset<len(groups[source]) and len(selected)<VALIDATION_GENERATION_LIMIT:
            selected.append(groups[source][offset]);added=True
    if not added: break
    offset+=1
VALIDATION_PROMPTS_PATH=PROJECT/'heldout_generation_prompts.json'
atomic_json(VALIDATION_PROMPTS_PATH,{p['id']:{'text':p['prompt']} for p in selected})
from rouge_score import rouge_scorer
from sacrebleu.metrics import BLEU
scorer=rouge_scorer.RougeScorer(['rougeL'],use_stemmer=False)
local_metric_rows=[];LOCAL_GENERATIONS={}

if RUN_GENERATION and RUN_HELDOUT_GENERATION:
    for plan in CANDIDATE_PLAN[:5]:
        checkpoint=FOUNDATION_CHECKPOINTS[plan['architecture']]
        output=candidate_output(plan,heldout=True)
        predictions=execute_generation(plan['variant'],VALIDATION_PROMPTS_PATH,output,checkpoint,plan['seed_offset'])
        LOCAL_GENERATIONS[plan['label']]=predictions
        candidates=[predictions[p['id']]['text'] for p in selected]
        references=[p['reference'] for p in selected]
        rouge=sum(scorer.score(r,c)['rougeL'].fmeasure for r,c in zip(references,candidates))/len(selected)
        bleu_metric=BLEU(tokenize='13a');bleu=bleu_metric.corpus_score(candidates,[references])
        metadata=json.loads((output/'generation_metadata.json').read_text())
        local_metric_rows.append({'label':plan['label'],'architecture':plan['architecture'],'variant':plan['variant'],
            'heldout_examples':len(selected),'ROUGE_L_F1_0_to_1':rouge,'SacreBLEU4_0_to_100':bleu.score,
            'seconds_including_launch':metadata['seconds_this_session'],
            'BLEU_signature':str(bleu_metric.get_signature())})
    local_metrics=pd.DataFrame(local_metric_rows);display(local_metrics)
    local_metrics.to_csv(PROJECT/'heldout_generation_metrics.csv',index=False)
    sample_rows=[{'source':p['source'],'prompt_tail':p['prompt'][-350:],
        'generated':LOCAL_GENERATIONS[PRIMARY_CANDIDATE][p['id']]['text'],'heldout_reference':p['reference']}
        for p in selected[:4]]
    display(pd.DataFrame(sample_rows))

## 20. Generate at least five distinct official submission candidates

All four foundation checkpoints participate in the initial five experiments. The extra decoder/focused experiment gives the fifth candidate. A canonical prediction-content hash excludes duplicates; changing a ZIP filename or its timestamp does not create a distinct prediction. Reserve experiments use alternative decoding settings or a recorded seed if duplicate content occurs. The notebook stops when it has the configured minimum, or clearly errors if the bounded reserve plan cannot produce enough distinct candidates.

Every archive contains only `prediction.json` at its root, with all original official ID strings and exactly 128 words per continuation. The five named candidate ZIPs are copied to `submission_candidates/` for convenient individual download. Upload each individual archive to the Phase-1 competition; the all-output archive is a backup, not a prediction submission. Accepted competition uploads and their scores must be recorded separately.

Your legacy decoder/balanced and decoder/focused results use their previous directories and unchanged base configs. They can be reused when checkpoint, tokenizer, prompts and decoding identities match. The primary root `submission.zip` is the decoder/balanced candidate by default; its content is intentionally the same as that one candidate, so it does not count as an additional distinct experiment.

In [ ]:
SUBMISSIONS=[];CANDIDATE_ATTEMPTS=[]
if RUN_GENERATION:
    if local_metric_rows:
        estimates=[{'label':r['label'],'rough_official_generation_hours':
            r['seconds_including_launch']/r['heldout_examples']*len(OFFICIAL_PROMPTS)/3600}
            for r in local_metric_rows]
        display(pd.DataFrame(estimates))
        print('Small-sample timing includes process launch and varies with prompt length.')
    catalog_dir=PROJECT/'submission_candidates';catalog_dir.mkdir(parents=True,exist_ok=True)
    for plan in CANDIDATE_PLAN:
        if len(SUBMISSIONS)>=MIN_SUBMISSION_CANDIDATES: break
        name=plan['architecture'];checkpoint=FOUNDATION_CHECKPOINTS[name]
        output=candidate_output(plan)
        predictions=execute_generation(plan['variant'],OFFICIAL_PROMPTS_PATH,output,checkpoint,plan['seed_offset'])
        archive=output/'submission.zip'
        copied=catalog_dir/(plan['label']+'.zip')
        candidate={**plan,'archive':str(copied),'source_archive':str(archive),
            'sha256':file_hash(archive),'checkpoint':str(checkpoint),'checkpoint_sha256':CHECKPOINT_HASHES[name],
            'decode':decode_config(plan['variant'],plan['seed_offset'])}
        accepted,duplicate_of=add_distinct_candidate(SUBMISSIONS,candidate,predictions)
        CANDIDATE_ATTEMPTS.append({**candidate,'included_as_distinct_candidate':accepted,'duplicate_of':duplicate_of})
        atomic_json(PROJECT/'submission_attempts.json',CANDIDATE_ATTEMPTS)
        if not accepted:
            print('Duplicate prediction content:',plan['label'],'matches',duplicate_of,'; trying a reserve experiment.')
            continue
        shutil.copyfile(archive,copied)
        atomic_json(PROJECT/'submission_candidates_partial_manifest.json',SUBMISSIONS)
        print('Ready:',plan['label'],'| distinct candidates:',len(SUBMISSIONS),'| IDs:',len(predictions))
        display(FileLink(str(copied)))
    assert len(SUBMISSIONS)>=MIN_SUBMISSION_CANDIDATES, 'Not enough distinct predictions; inspect generation/model quality and add a substantive experiment.'
    primary=next(r for r in SUBMISSIONS if r['label']==PRIMARY_CANDIDATE)
    primary_predictions=json.loads(Path(primary['source_archive']).with_name('prediction.json').read_text())
    write_submission(primary_predictions,OFFICIAL_PROMPTS,PROJECT)
    atomic_json(PROJECT/'submission_manifest.json',{'primary':PRIMARY_CANDIDATE,'candidates':SUBMISSIONS,
        'foundation_models':foundation_rows,'tokenizer_sha256':TOKENIZER_SHA256,'dataset_revision':REVISION,
        'official_test_path':test_path,'official_example_count':len(OFFICIAL_PROMPTS)})
    display(pd.DataFrame([{k:r[k] for k in ['label','architecture','variant','examples','archive']} for r in SUBMISSIONS]))
    display(FileLink(str(PROJECT/'submission.zip')))
else:
    print('Generation disabled. No new official candidates were produced.')

## 21. Independent final readback

This readback checks the actual bytes in each archive rather than relying on a successful write call. After downloading a submission, upload the zip for the **Phase-1** Codabench window and record whether it was accepted. The document lists a maximum of 10 submissions/day and 40/phase, and four distinct valid Phase-1 submissions as a minimum. Retain scoring evidence; merely producing four local zip files is not four valid submissions.

The notebook does not register accounts or upload to Codabench automatically. It produces the complete reviewable files. Use the required username format from the document: first letter of name, underscore, roll number.

In [ ]:
import zipfile
if RUN_GENERATION:
    fingerprints=set()
    for row in SUBMISSIONS:
        with zipfile.ZipFile(row['archive']) as archive:
            assert archive.namelist()==['prediction.json']
            data=json.loads(archive.read('prediction.json').decode('utf-8'))
        assert validate_predictions(data,OFFICIAL_PROMPTS)
        fingerprint=prediction_fingerprint(data)
        assert fingerprint==row['prediction_fingerprint'] and fingerprint not in fingerprints
        fingerprints.add(fingerprint)
        counts=[len(v['text'].split()) for v in data.values()]
        print(row['label'],'| IDs:',len(data),'| words:',min(counts),max(counts),'| ZIP root valid')
    assert len(fingerprints)>=MIN_SUBMISSION_CANDIDATES>=5
    print('At least five distinct local candidate archives validated. Record Codabench acceptance after uploading.')

## 22. Experiment/report evidence and LLM-use declaration

Fill in your observations after your own runs. The CSVs contain measurements; they are not a completed analysis or a fabricated leaderboard result. The report should explain why the architecture, tokenizer, corpus mix, context, schedule and decoding choices were made, then compare their actual effects on loss, generation quality, speed and memory. Note confounders such as early time stops, different token targets or a changed validation protocol.

Useful questions to answer:

1. Which architecture gives the best quality for the same target-token budget? Does the conclusion survive a larger budget?
2. How much slower is the recurrent component? Does its validation or generation behavior justify that cost?
3. Does local attention lose useful distant citations or gain regularization? Does GQA reduce inference-cache cost in your measurements?
4. How do token/word ratios vary for statutes, judgments and the official prompts? What fraction of generation prompts exceeds training context length?
5. Is the best validation checkpoint also your best generation checkpoint? Are the score differences reproducible with fixed seeds?
6. Which decoding setting preserves names/numbers, avoids loops and reads naturally on heldout examples? What are its ROUGE/BLEU and official BERTScore values?
7. How much training time is spent outside compute, including data preparation, validation, checkpoints and generation?

The declaration draft below records the actual extent of coding help, including the custom architectures. Adapt it to any official declaration form the course provides. Preserve the professor's update yourself as part of your course record; the Discord message was supplied by you, not independently fetched here.

In [ ]:
declaration="""LLM-use declaration draft - CS-781 Capstone Phase 1

Student: Hardik Tiwari
Assistance: ChatGPT/Codex was used to create this Kaggle notebook, including data
processing, custom model architectures, training, inference, tests, documentation,
and research-source summaries. The professor's coding-permission update was
provided by the student: "You can use LLM for coding the assignment."

The student is responsible for reviewing and understanding the code, obtaining
authorized official dataset access, running the experiments, analyzing measured
results, writing conclusions, and complying with the current competition rules.
No pretrained language-model weights or external model APIs are used here.

Replace this draft with/adapt it to the official course declaration form when required.
Add run dates, notebook version, and any later assistance to the final declaration.
"""
(PROJECT/'LLM_use_declaration.txt').write_text(declaration)
report_template="""# Phase-1 experiment notes

## Data and tokenizer
Record source revision, source token mix, unique cached tokens, holdout construction,
duplicate handling, tokenizer vocabulary and tokens/word statistics.

## Architecture experiments
Use foundation_experiments.csv and, if enabled, architecture_experiments.csv. Record the actual target tokens and time caps.
Explain information flow, parameter counts and any unequal-budget limitations.

## All foundation runs and context experiments
Record random/own-pilot initialization, objective, effective batch, LR schedule,
processed tokens, chosen checkpoint and reason for selection.

## Efficiency evidence
Use benchmark_summary.json and environment.json. Record both memory and target
tokens/s, and include non-training overhead in end-to-end runtime claims.

## Generation and submissions
Compare heldout examples and overlap diagnostics. Record accepted Codabench
submission IDs, official BERTScore/ROUGE-L/BLEU, checkpoint and decoding config.

## Conclusions and limitations
Write your own conclusions from measurements. Distinguish pilot results from
full-budget results and acknowledge validation/generation differences.

## References and LLM use
Cite the sources listed in the notebook and attach the required declaration.
"""
(PROJECT/'experiment_notes.md').write_text(report_template)
submission_record=pd.DataFrame([{'label':r['label'],'architecture':r['architecture'],'variant':r['variant'],'local_archive_sha256':r['sha256'],
    'codabench_submission_id':'','accepted':'','BERTScore_F1':'','ROUGE_L_F1':'','BLEU_4':'',
    'uploaded_at_with_timezone':''} for r in SUBMISSIONS])
record_path=PROJECT/'codabench_submission_record.csv'
if record_path.exists():
    previous_record=pd.read_csv(record_path).fillna('')
    for index,row in submission_record.iterrows():
        matches=previous_record[previous_record.local_archive_sha256==row.local_archive_sha256]
        if len(matches):
            for column in ['codabench_submission_id','accepted','BERTScore_F1','ROUGE_L_F1','BLEU_4','uploaded_at_with_timezone']:
                submission_record.at[index,column]=matches.iloc[0][column]
submission_record.to_csv(record_path,index=False)
display(submission_record)

## 23. Save the foundation for Phases 2-7

Keep `best_model.pt`, the exact tokenizer and model source together. Future tasks must start from your own Phase-1 or later-phase model according to the source document. For the main decoder, `model(input_ids)` returns contextual hidden states of shape `[batch, tokens, 512]`; a future classification/entity/role head can use these representations. `model.project(hidden)` is the tied language-model head. The encoder-decoder exposes `model.encode(source_ids)` for prefix/document representations.

The inference bundle contains selected weights, tokenizer, source modules, metadata, research references, logs/experiment summaries and the declaration. It excludes giant recovery/cache files to avoid duplicating them. **Keep/download the recovery run folder and cache separately** if you intend to resume Phase-1 optimizer training. An inference bundle alone is enough to load the foundation for a new adaptation stage but is not an exact optimizer resume.

In [ ]:
requirements="""torch>=2.3
tokenizers>=0.20,<0.24
huggingface_hub>=0.26,<2
ijson>=3.2,<4
requests>=2.31,<3
sacrebleu>=2.4,<3
rouge-score>=0.1.2,<0.2
"""
(PROJECT/'requirements.txt').write_text(requirements)
readme=f"""CS-781 Phase-1: all four legal foundations
Architectures: {', '.join(FOUNDATION_ARCHITECTURES)}
Primary foundation: {MAIN_ARCHITECTURE}
Dataset: {REPO}, revision {REVISION}
Tokenizer SHA256: {TOKENIZER_SHA256}

Each foundations/<architecture>/best_model.pt is your own trained checkpoint.
Install a compatible CUDA-enabled PyTorch and requirements.txt.
Load only trusted own checkpoints with torch.load(..., map_location='cpu', weights_only=False).
Instantiate LegalLM(ModelConfig(**checkpoint['model_config'])); load checkpoint['model'].
Use the unchanged tokenizer vocabulary IDs. Encoder-decoder requires source_ids.
For exact training recovery also keep latest.pt, tokenizer, cache and run configuration.
The compact foundation bundle excludes large recovery caches/latest optimizer states;
the final all_kaggle_outputs.zip retains the entire working tree.

submission_candidates/ contains individual competition archives. Each candidate ZIP
contains only prediction.json at its root. Upload individual ZIPs, not this bundle.
Generating at least five distinct candidates does not establish competition acceptance.
"""
(PROJECT/'README.txt').write_text(readme)
bundle=PROJECT/'phase1_all_foundations_bundle.zip'
support=['legal_models.py','legal_data.py','train_phase1.py','generate_phase1.py','foundation_workflow.py',
    'tokenizer.json','tokenizer.provenance.json','requirements.txt','README.txt','LLM_use_declaration.txt',
    'experiment_notes.md','research_references.md','benchmark_summary.json','foundation_experiments.csv',
    'foundation_manifest.json','architecture_experiments.csv','heldout_generation_metrics.csv',
    'submission_manifest.json','submission_attempts.json','codabench_submission_record.csv',
    'data_access_metadata.json','training_curves_all_foundations.png']
with zipfile.ZipFile(bundle,'w',compression=zipfile.ZIP_STORED,allowZip64=True) as archive:
    for name in support:
        path=PROJECT/name
        if path.exists(): archive.write(path,arcname=name)
    archive.write(CACHE_DIR/'manifest.json',arcname='cache_manifest.json')
    for name in FOUNDATION_ARCHITECTURES:
        archive.write(FOUNDATION_CHECKPOINTS[name],arcname=f'foundations/{name}/best_model.pt')
        for filename in ['run_config.json','environment.json','best_metrics.json','training_summary.json','train_log.csv','validation_log.jsonl']:
            path=FOUNDATION_RUNS[name]/filename
            if path.exists(): archive.write(path,arcname=f'foundations/{name}/{filename}')
    for row in SUBMISSIONS:
        archive.write(row['archive'],arcname='submission_candidates/'+Path(row['archive']).name)
print('All-foundation bundle:',round(bundle.stat().st_size/2**20,2),'MiB')
display(FileLink(str(bundle)))

## 24. Troubleshooting and controlled follow-up experiments

| Symptom | Concrete next action |
|---|---|
| HTTP 401/403 for official data | Obtain dataset access in your own HF account; authorize `HF_TOKEN` in Kaggle Secrets. Or attach the official Phase-1 files locally. |
| Schema parser finds no text | Inspect one official record; set `SOURCE_LAYOUT` and/or extend `TEXT_KEYS` for its actual documented field. Do not concatenate IDs/labels as legal text. |
| Tokenizer has fewer than 32,768 entries | Increase training characters per source and retrain. Keep validation excluded. |
| CUDA OOM during preflight | Reduce micro-batch, increase accumulation, optionally enable block/loss checkpointing, then rerun the actual two-GPU benchmark. |
| NCCL/DDP launch fails | Confirm two visible CUDA GPUs and a recent CUDA PyTorch image; rerun the short two-GPU preflight. A one-GPU fallback requires explicitly enabling `ALLOW_SINGLE_GPU` and a fresh batch-aware config. |
| AMP repeatedly skips updates | Inspect logs/gradients/data; lower LR; use a short FP32 diagnostic. Do not call a run trained if its updates overflowed. |
| Validation is noisy | Use a larger fixed heldout subset and more generation pairs; compare runs on the same split/tokenizer/objective/context. |
| `latest.pt` cannot resume | Restore its matching tokenizer/cache/best weights and keep the saved architecture, world size, batch geometry and schedule. Use own weights via `--init-from` for a deliberate new stage. |
| Generation reaches the token cap | Use the included length-recovery generator; retain unchanged base decode configs for partial reuse; inspect per-ID retry audits. Do not pad with copied/filler words. |
| A prompt exceeds 2048 tokens | Raise the explicit prompt/position limits while preserving the full prompt; consider longer-context training and record extrapolation limitations. |
| Candidate archives are identical | Change a real decoding or own-checkpoint experiment, regenerate, and record what changed. Renaming a zip does not count. |
| Good NLL but weak BERTScore | Inspect heldout continuations; compare checkpoint choice/decoding and a larger training-token budget. NLL is not the competition metric. |
| A foundation run is too slow for remaining time | Use measured throughput, shorten the token target before a new run, reduce expensive validation events, and reserve time for generation/uploads/scoring. |

Useful additional ablations: source mixes; 512 vs 1024 vs 2048 training context; equal wall-time vs equal supervised-token architecture budgets; gradient checkpointing on/off with a fixed effective batch; validation-selected checkpoint vs final checkpoint; greedy vs nucleus decoding. Change one dimension at a time and record the actual resource/quality change. Optional compression should use only your own trained teacher/model and remain within the current course constraints.

## 25. Research references and implementation provenance

The architecture and training source in this notebook is a custom PyTorch implementation. The references below motivate components and optimizations; the notebook does not copy a released model's weights or claim to reproduce published benchmark results.

1. Micikevicius et al. (2017), **Mixed Precision Training**. [arXiv:1710.03740](https://arxiv.org/abs/1710.03740).
2. Li et al. (2020), **PyTorch Distributed: Experiences on Accelerating Data Parallel Training**. [arXiv:2006.15704](https://arxiv.org/abs/2006.15704).
3. Dao et al. (2022), **FlashAttention: Fast and Memory-Efficient Exact Attention with IO-Awareness**. [arXiv:2205.14135](https://arxiv.org/abs/2205.14135).
4. Dao (2023), **FlashAttention-2: Faster Attention with Better Parallelism and Work Partitioning**. [arXiv:2307.08691](https://arxiv.org/abs/2307.08691). [Official implementation/hardware notes](https://github.com/Dao-AILab/flash-attention).
5. Zhang & Sennrich (2019), **Root Mean Square Layer Normalization**. [arXiv:1910.07467](https://arxiv.org/abs/1910.07467).
6. Ainslie et al. (2023), **GQA: Training Generalized Multi-Query Transformer Models from Multi-Head Checkpoints**. [arXiv:2305.13245](https://arxiv.org/abs/2305.13245). The notebook trains its GQA model from scratch rather than applying the paper's checkpoint-conversion recipe.
7. Chen et al. (2016), **Training Deep Nets with Sublinear Memory Cost**. [arXiv:1604.06174](https://arxiv.org/abs/1604.06174).
8. Hoffmann et al. (2022), **Training Compute-Optimal Large Language Models**. [arXiv:2203.15556](https://arxiv.org/abs/2203.15556).
9. Su et al. (2021), **RoFormer: Enhanced Transformer with Rotary Position Embedding**. [arXiv:2104.09864](https://arxiv.org/abs/2104.09864).
10. Shazeer (2020), **GLU Variants Improve Transformer**. [arXiv:2002.05202](https://arxiv.org/abs/2002.05202).
11. PyTorch, [SDPA API](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.scaled_dot_product_attention.html), [AMP examples](https://docs.pytorch.org/docs/stable/notes/amp_examples.html), [DDP API](https://docs.pytorch.org/docs/stable/generated/torch.nn.parallel.DistributedDataParallel.html), and [Performance Tuning Guide](https://docs.pytorch.org/tutorials/recipes/recipes/tuning_guide.html).
12. Course source: [Capstone Document](https://docs.google.com/document/d/1olqoT5KPhf5CSJLBNjXGyKyyyQDBr6jcnWqPUh5cNk4/edit), through printed page 17, read 5 October 2026; [official dataset](https://huggingface.co/datasets/Exploration-Lab/CS-781-Capstone).

Record the library versions, GPU configuration and dataset revision from your actual run. Kernel availability and speed depend on that environment, so the included preflight measurements are part of the experiment rather than an optional assumed result.

In [ ]:
references="""# Research sources used in the Phase-1 notebook
- Mixed precision: https://arxiv.org/abs/1710.03740
- PyTorch DDP: https://arxiv.org/abs/2006.15704
- FlashAttention: https://arxiv.org/abs/2205.14135
- FlashAttention-2: https://arxiv.org/abs/2307.08691
- Official hardware support: https://github.com/Dao-AILab/flash-attention
- RMSNorm: https://arxiv.org/abs/1910.07467
- GQA: https://arxiv.org/abs/2305.13245
- Activation checkpointing: https://arxiv.org/abs/1604.06174
- Compute-optimal training: https://arxiv.org/abs/2203.15556
- RoPE: https://arxiv.org/abs/2104.09864
- SwiGLU: https://arxiv.org/abs/2002.05202
- SDPA: https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.scaled_dot_product_attention.html
- AMP: https://docs.pytorch.org/docs/stable/notes/amp_examples.html
- DDP: https://docs.pytorch.org/docs/stable/generated/torch.nn.parallel.DistributedDataParallel.html
- Performance tuning: https://docs.pytorch.org/tutorials/recipes/recipes/tuning_guide.html
- Official dataset: https://huggingface.co/datasets/Exploration-Lab/CS-781-Capstone
- Course brief: https://docs.google.com/document/d/1olqoT5KPhf5CSJLBNjXGyKyyyQDBr6jcnWqPUh5cNk4/edit
"""
(PROJECT/'research_references.md').write_text(references)
# Final refresh adds the just-written references without duplicating any zip entry.
with zipfile.ZipFile(bundle,'a',compression=zipfile.ZIP_STORED) as archive:
    if 'research_references.md' not in archive.namelist():
        archive.write(PROJECT/'research_references.md',arcname='research_references.md')
print('Selected foundation:',FINAL_CHECKPOINT)
print('Distinct submission candidates:',len(SUBMISSIONS))
display(FileLink(str(bundle)))
if RUN_GENERATION: display(FileLink(str(PROJECT/'submission.zip')))

## 26. Download all Kaggle working outputs

Run this final cell after generation/export finishes. It archives every file under `/kaggle/working`, including checkpoints, caches, source, logs, and submission archives. Paths remain relative to that directory. The output archive and its temporary file are excluded, so rerunning does not archive the previous all-output ZIP. `ZIP_STORED` avoids recompressing large model/data files. Download the resulting ZIP from the displayed link or the notebook Output panel.

For your existing Kaggle session, paste just this final code cell; keep the installed generation recovery patch.


In [ ]:
from pathlib import Path
import shutil
import zipfile
from IPython.display import display, FileLink

WORKING = Path("/kaggle/working")
OUTPUT_ZIP = WORKING / "all_kaggle_outputs.zip"
TEMP_ZIP = WORKING / "all_kaggle_outputs.zip.tmp"

excluded = {OUTPUT_ZIP.resolve(), TEMP_ZIP.resolve()}
files = sorted(p for p in WORKING.rglob("*")
               if p.is_file() and p.resolve() not in excluded)
total_bytes = sum(p.stat().st_size for p in files)
if shutil.disk_usage(WORKING).free < total_bytes + 64 * 1024**2:
    raise RuntimeError("Not enough free space to ZIP all outputs. "
                       f"Need approximately {total_bytes / 1024**3:.2f} GiB of additional space.")

try:
    with zipfile.ZipFile(TEMP_ZIP, "w", compression=zipfile.ZIP_STORED,
                         allowZip64=True) as archive:
        for index, path in enumerate(files, 1):
            archive.write(path, arcname=path.relative_to(WORKING).as_posix())
            if index % 100 == 0 or index == len(files):
                print(f"Archived {index}/{len(files)} files", flush=True)
    TEMP_ZIP.replace(OUTPUT_ZIP)
except Exception:
    TEMP_ZIP.unlink(missing_ok=True)
    raise

print(f"Saved: {OUTPUT_ZIP}")
print(f"Size: {OUTPUT_ZIP.stat().st_size / 1024**3:.2f} GiB")
display(FileLink(str(OUTPUT_ZIP)))
